# Clustering Presupuestal Municipal (2022–2024) — Versión Optimizada

Este notebook realiza un análisis completo de clustering presupuestal de municipalidades peruanas con **tres enfoques complementarios**:

## 📊 Metodologías Implementadas:

### 1. **Análisis Estático (K-Means)**
Agrupa municipios según indicadores presupuestales **sin considerar el tiempo**. Proporciona una visión global del desempeño presupuestal mediante clustering tradicional K-Means.

**Aplicación**: Benchmarking general y clasificación básica de municipalidades.

### 2. **Análisis de Panel**
Análisis longitudinal que **considera la estructura temporal** realizando clustering separado para cada año (2022, 2023, 2024). Permite observar la evolución de los centroides año a año.

**Aplicación**: Políticas que requieren ajustes anuales y análisis de tendencias temporales.

### 3. **Análisis de Trayectorias**
**Seguimiento de cambios** en asignación de clusters a lo largo del tiempo. Identifica patrones como:
- **ESTABLE**: Municipalidades que permanecen en el mismo cluster
- **MEJORANDO**: Transición hacia clusters de mejor desempeño
- **EMPEORANDO**: Transición hacia clusters de menor desempeño  
- **FLUCTUANTE**: Cambios variables sin patrón claro

**Aplicación**: Identificar municipalidades que progresan o retroceden para intervenciones focalizadas.

## 🎯 Indicadores Analizados:
- **ind_eje**: Indicador de ejecución presupuestal (0.0 - 1.0)
- **propim**: Proporción de inversión municipal (0.0 - 1.0)
- **proinv**: Proporción de inversión total (0.0 - 1.0)

## 📈 Resultados Generados:

### Gráficos Exploratorios (5)
1. Distribución de indicadores presupuestales
2. Análisis de variabilidad (boxplots)
3. Relaciones entre indicadores (scatter plots)
4. Matriz de correlación
5. Evolución temporal 2022-2024

### Análisis Estático (3)
6. Método del codo
7. Visualización PCA de clusters
8. Métricas de validación (Silhouette, Calinski-Harabasz, Davies-Bouldin)

### Análisis por Cluster (5)
9. Características promedio por cluster
10. Boxplots comparativos por cluster
11. Perfiles de radar (spider chart)
12. Distribución de tamaño de clusters
13. Comparación cluster vs población general

### Análisis de Panel (1)
14. Evolución de clusters por año

### Análisis de Trayectorias (1)
15. Matrices de transición y categorías

### Comparación de Métodos (1)
16. Análisis comparativo de los 3 enfoques

## 📄 Documentación:
- **Informe Word**: Documento completo con todos los análisis, gráficos y comparación entre métodos
- **Datasets CSV**: Resultados de clustering con asignaciones por cada método
- **Gráficos PNG**: 16 visualizaciones en alta resolución (300 DPI)

---

**Autor**: Análisis automatizado  
**Fecha**: 2025  
**Dataset**: 5,305 registros de 1,770 municipalidades peruanas  
**Periodo**: 2022-2024

In [ ]:
# @title 1. Instalación de Librerías Necesarias
print("Instalando librerías necesarias...")

# Instalar librerías si no están disponibles
try:
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    import seaborn as sns
    from sklearn.cluster import KMeans
    from sklearn.preprocessing import StandardScaler
    from sklearn.decomposition import PCA
    print("✓ Todas las librerías están disponibles")
except ImportError as e:
    print(f"Instalando librería faltante: {e}")
    !pip install pandas numpy matplotlib seaborn scikit-learn -q
    print("✓ Instalación completada")

# Configuración de visualización
import warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("\n✓ Configuración completada")

In [ ]:
# @title 2. Carga del Archivo CSV 📁
import pandas as pd
import os

print("=" * 60)
print("CARGA DE DATOS - CLUSTERING PRESUPUESTAL MUNICIPAL")
print("=" * 60)

# Detectar si estamos en Google Colab
try:
    import google.colab
    IN_COLAB = True
except:
    IN_COLAB = False

# Opción 1: Si estamos en Google Colab, permitir subir archivo
if IN_COLAB:
    print("\n📌 Entorno detectado: Google Colab")
    print("\nOpciones de carga:")
    print("1. Subir archivo manualmente")
    print("2. Cargar desde Google Drive")
    print("3. Usar archivo del repositorio (si está clonado)")
    
    opcion = input("\nSeleccione opción (1/2/3): ").strip()
    
    if opcion == "1":
        from google.colab import files
        print("\n📤 Por favor, sube el archivo 'base.csv'")
        uploaded = files.upload()
        csv_filename = list(uploaded.keys())[0]
        print(f"✓ Archivo '{csv_filename}' cargado exitosamente")
        
    elif opcion == "2":
        from google.colab import drive
        drive.mount('/content/drive')
        print("\n📁 Google Drive montado")
        csv_path = input("Ingrese la ruta del archivo CSV en Drive (ej: /content/drive/MyDrive/base.csv): ")
        csv_filename = csv_path
        
    elif opcion == "3":
        csv_filename = "base.csv"
        if not os.path.exists(csv_filename):
            print(f"⚠️ Archivo '{csv_filename}' no encontrado en el directorio actual")
            print("Por favor, clone el repositorio o suba el archivo manualmente")
        else:
            print(f"✓ Usando archivo local: {csv_filename}")
else:
    # Opción 2: Si estamos en entorno local
    print("\n📌 Entorno detectado: Local/Jupyter")
    csv_filename = "base.csv"
    
    if os.path.exists(csv_filename):
        print(f"✓ Archivo '{csv_filename}' encontrado en el directorio actual")
    else:
        print(f"⚠️ Archivo '{csv_filename}' no encontrado")
        print("Asegúrese de que el archivo esté en el mismo directorio que este notebook")

# Cargar el archivo CSV
try:
    print(f"\n🔄 Cargando datos desde '{csv_filename}'...")
    df = pd.read_csv(csv_filename)
    print(f"✅ Datos cargados exitosamente!")
    print(f"\n📊 Dimensiones del dataset: {df.shape[0]} filas × {df.shape[1]} columnas")
    print(f"📋 Columnas: {', '.join(df.columns.tolist())}")
    
    # Mostrar información básica
    print("\n" + "=" * 60)
    print("INFORMACIÓN DEL DATASET")
    print("=" * 60)
    print(df.info())
    
    print("\n" + "=" * 60)
    print("PRIMERAS 5 FILAS")
    print("=" * 60)
    print(df.head())
    
except FileNotFoundError:
    print(f"❌ ERROR: No se pudo encontrar el archivo '{csv_filename}'")
    print("Por favor, verifique la ruta del archivo")
except Exception as e:
    print(f"❌ ERROR al cargar el archivo: {str(e)}")

In [ ]:
# @title 4. Visualizaciones Exploratorias 📈
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Paleta de colores profesional
COLOR_PRIMARY = '#2C3E50'    # Azul oscuro corporativo
COLOR_SECONDARY = '#3498DB'  # Azul medio
COLOR_ACCENT = '#E74C3C'     # Rojo acento
COLOR_SUCCESS = '#27AE60'    # Verde
COLOR_WARNING = '#F39C12'    # Naranja
COLOR_INFO = '#9B59B6'       # Púrpura

# Configurar estilo
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette([COLOR_PRIMARY, COLOR_SECONDARY, COLOR_SUCCESS, COLOR_INFO])

print("Generando visualizaciones exploratorias individuales...")

indicadores = ['ind_eje', 'propim', 'proinv']
nombres_indicadores = {
    'ind_eje': 'Indicador de Ejecución Presupuestal',
    'propim': 'Proporción de Inversión Municipal',
    'proinv': 'Proporción de Inversión Total'
}

# ========== GRÁFICO 1: Distribuciones (Histogramas) ==========
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Distribución de Indicadores Presupuestales', fontsize=16, fontweight='bold', y=1.02)

for i, ind in enumerate(indicadores):
    axes[i].hist(df[ind], bins=40, alpha=0.7, color=COLOR_SECONDARY, edgecolor=COLOR_PRIMARY, linewidth=1.2)
    axes[i].set_title(nombres_indicadores[ind], fontsize=12, fontweight='bold')
    axes[i].set_xlabel('Valor', fontsize=10)
    axes[i].set_ylabel('Frecuencia', fontsize=10)
    axes[i].grid(alpha=0.3, linestyle='--')
    axes[i].axvline(df[ind].mean(), color=COLOR_ACCENT, linestyle='--', linewidth=2, label=f'Media: {df[ind].mean():.3f}')
    axes[i].legend()

plt.tight_layout()
plt.savefig('01_distribucion_indicadores.png', dpi=300, bbox_inches='tight', facecolor='white')
print("✓ Guardado: 01_distribucion_indicadores.png")
plt.close()

# ========== GRÁFICO 2: Boxplots Comparativos ==========
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Análisis de Variabilidad - Boxplots', fontsize=16, fontweight='bold', y=1.02)

for i, ind in enumerate(indicadores):
    bp = axes[i].boxplot(df[ind].dropna(), vert=True, patch_artist=True,
                         boxprops=dict(facecolor=COLOR_SECONDARY, alpha=0.7),
                         medianprops=dict(color=COLOR_ACCENT, linewidth=2),
                         whiskerprops=dict(color=COLOR_PRIMARY, linewidth=1.5),
                         capprops=dict(color=COLOR_PRIMARY, linewidth=1.5))
    axes[i].set_title(nombres_indicadores[ind], fontsize=12, fontweight='bold')
    axes[i].set_ylabel('Valor', fontsize=10)
    axes[i].grid(alpha=0.3, axis='y', linestyle='--')

plt.tight_layout()
plt.savefig('02_boxplots_indicadores.png', dpi=300, bbox_inches='tight', facecolor='white')
print("✓ Guardado: 02_boxplots_indicadores.png")
plt.close()

# ========== GRÁFICO 3: Scatter Plots (Relaciones) ==========
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Relaciones entre Indicadores', fontsize=16, fontweight='bold', y=1.02)

# ind_eje vs propim
axes[0].scatter(df['ind_eje'], df['propim'], alpha=0.4, s=20, color=COLOR_SECONDARY, edgecolors=COLOR_PRIMARY, linewidth=0.5)
axes[0].set_xlabel('Indicador de Ejecución', fontsize=10)
axes[0].set_ylabel('Proporción Inversión Municipal', fontsize=10)
axes[0].set_title('Ejecución vs Inversión Municipal', fontsize=12, fontweight='bold')
axes[0].grid(alpha=0.3, linestyle='--')

# ind_eje vs proinv
axes[1].scatter(df['ind_eje'], df['proinv'], alpha=0.4, s=20, color=COLOR_SUCCESS, edgecolors=COLOR_PRIMARY, linewidth=0.5)
axes[1].set_xlabel('Indicador de Ejecución', fontsize=10)
axes[1].set_ylabel('Proporción Inversión Total', fontsize=10)
axes[1].set_title('Ejecución vs Inversión Total', fontsize=12, fontweight='bold')
axes[1].grid(alpha=0.3, linestyle='--')

# propim vs proinv
axes[2].scatter(df['propim'], df['proinv'], alpha=0.4, s=20, color=COLOR_INFO, edgecolors=COLOR_PRIMARY, linewidth=0.5)
axes[2].set_xlabel('Proporción Inversión Municipal', fontsize=10)
axes[2].set_ylabel('Proporción Inversión Total', fontsize=10)
axes[2].set_title('Inversión Municipal vs Total', fontsize=12, fontweight='bold')
axes[2].grid(alpha=0.3, linestyle='--')

plt.tight_layout()
plt.savefig('03_scatter_relaciones.png', dpi=300, bbox_inches='tight', facecolor='white')
print("✓ Guardado: 03_scatter_relaciones.png")
plt.close()

# ========== GRÁFICO 4: Matriz de Correlación ==========
fig, ax = plt.subplots(figsize=(8, 6))
corr_matrix = df[indicadores].corr()

# Crear heatmap con colores profesionales
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
cmap = sns.diverging_palette(250, 10, as_cmap=True)  # Azul a Rojo
sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap=cmap, 
            square=True, linewidths=2, cbar_kws={"shrink": 0.8},
            annot_kws={'size': 12, 'weight': 'bold'},
            vmin=-1, vmax=1, center=0,
            mask=mask)

plt.title('Matriz de Correlación entre Indicadores', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('04_matriz_correlacion.png', dpi=300, bbox_inches='tight', facecolor='white')
print("✓ Guardado: 04_matriz_correlacion.png")
plt.close()

# ========== GRÁFICO 5: Evolución Temporal ==========
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Evolución Temporal de Indicadores (2022-2024)', fontsize=16, fontweight='bold', y=1.02)

promedios_anuales = df.groupby('year')[indicadores].mean()

for i, ind in enumerate(indicadores):
    axes[i].plot(promedios_anuales.index, promedios_anuales[ind], 
                marker='o', markersize=10, linewidth=3, color=COLOR_SECONDARY,
                markerfacecolor=COLOR_PRIMARY, markeredgewidth=2, markeredgecolor='white')
    axes[i].set_title(nombres_indicadores[ind], fontsize=12, fontweight='bold')
    axes[i].set_xlabel('Año', fontsize=10)
    axes[i].set_ylabel('Promedio', fontsize=10)
    axes[i].grid(alpha=0.3, linestyle='--')
    axes[i].set_ylim([promedios_anuales[ind].min() * 0.95, promedios_anuales[ind].max() * 1.05])
    
    # Añadir valores
    for year, val in zip(promedios_anuales.index, promedios_anuales[ind]):
        axes[i].text(year, val, f'{val:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('05_evolucion_temporal.png', dpi=300, bbox_inches='tight', facecolor='white')
print("✓ Guardado: 05_evolucion_temporal.png")
plt.close()

print("\n✓ Todas las visualizaciones exploratorias generadas exitosamente")
print("  Total de gráficos: 5")

In [ ]:
# @title 6. Visualización de Clusters con PCA 🎨
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

print("Generando visualización de clusters con PCA...")

# Verificar si las variables necesarias existen, si no, recalcularlas
if 'X_scaled' not in locals() or 'clusters' not in locals() or 'kmeans' not in locals():
    print("Recalculando datos necesarios...")
    indicadores = ['ind_eje', 'propim', 'proinv']
    X = df[indicadores].copy()
    X_clean = X.dropna()
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_clean)
    
    from sklearn.cluster import KMeans
    k_optimo = 3
    kmeans = KMeans(n_clusters=k_optimo, random_state=42, n_init=10)
    clusters = kmeans.fit_predict(X_scaled)
    print("✓ Datos recalculados")

# Aplicar PCA para reducir a 2 dimensiones
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

# Crear visualización
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Gráfico 1: Clusters con PCA
scatter = axes[0].scatter(X_pca[:, 0], X_pca[:, 1], 
                          c=clusters, cmap='viridis', 
                          alpha=0.6, s=50, edgecolors='black', linewidth=0.5)
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%} varianza)', fontsize=12)
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%} varianza)', fontsize=12)
axes[0].set_title('Clusters K-Means (Proyección PCA)', fontsize=14, fontweight='bold')
axes[0].grid(alpha=0.3)

# Agregar centroides
centroides_pca = pca.transform(kmeans.cluster_centers_)
axes[0].scatter(centroides_pca[:, 0], centroides_pca[:, 1], 
                c='red', marker='X', s=300, edgecolors='black', 
                linewidth=2, label='Centroides')
axes[0].legend(fontsize=10)

# Agregar colorbar
cbar = plt.colorbar(scatter, ax=axes[0])
cbar.set_label('Cluster', fontsize=11)

# Gráfico 2: Varianza explicada por componentes principales
explained_var = pca.explained_variance_ratio_
axes[1].bar(['PC1', 'PC2'], explained_var, color=['steelblue', 'coral'], alpha=0.7)
axes[1].set_ylabel('Varianza Explicada', fontsize=12)
axes[1].set_title('Varianza Explicada por Componentes Principales', fontsize=14, fontweight='bold')
axes[1].grid(alpha=0.3, axis='y')

# Agregar texto con varianza total
total_var = sum(explained_var)
axes[1].text(0.5, max(explained_var) * 0.9, 
             f'Varianza Total Explicada: {total_var:.2%}',
             ha='center', fontsize=11, fontweight='bold',
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.savefig('clusters_pca_visualization.png', dpi=300, bbox_inches='tight')
print("✓ Visualización guardada en 'clusters_pca_visualization.png'")
plt.show()

# Matriz de componentes principales
print("\n📊 CONTRIBUCIÓN DE VARIABLES A COMPONENTES PRINCIPALES:")
components_df = pd.DataFrame(
    pca.components_.T,
    columns=['PC1', 'PC2'],
    index=indicadores
)
print(components_df)
print(f"\n✓ Varianza total explicada por PC1 y PC2: {total_var:.2%}")

In [ ]:
# @title 6.5. Pruebas Estadísticas y Descripción de Clusters 📏
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("=" * 60)
print("PRUEBAS ESTADÍSTICAS Y VALIDACIÓN DE CLUSTERING")
print("=" * 60)

# Asegurar que tenemos las variables necesarias
if 'X_scaled' not in locals() or 'clusters' not in locals():
    print("Recalculando datos...")
    indicadores = ['ind_eje', 'propim', 'proinv']
    X = df[indicadores].copy()
    X_clean = X.dropna()
    from sklearn.preprocessing import StandardScaler
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_clean)
    from sklearn.cluster import KMeans
    kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
    clusters = kmeans.fit_predict(X_scaled)
    df_clustered = X_clean.copy()
    df_clustered['cluster_estatico_nuevo'] = clusters

# ============================================================
# PRUEBAS ESTADÍSTICAS DE VALIDACIÓN
# ============================================================

print("\n📊 MÉTRICAS DE CALIDAD DEL CLUSTERING:\n")

# 1. Silhouette Score (rango: -1 a 1, mejor cerca de 1)
silhouette = silhouette_score(X_scaled, clusters)
print(f"1️⃣ Silhouette Score: {silhouette:.4f}")
print(f"   Interpretación: ", end="")
if silhouette > 0.7:
    print("Excelente - Clusters muy bien definidos")
elif silhouette > 0.5:
    print("Bueno - Estructura de clusters razonable")
elif silhouette > 0.25:
    print("Aceptable - Cierta estructura de clusters presente")
else:
    print("Débil - Clusters poco definidos")

# 2. Calinski-Harabasz Index (mayor es mejor)
calinski = calinski_harabasz_score(X_scaled, clusters)
print(f"\n2️⃣ Calinski-Harabasz Index: {calinski:.4f}")
print(f"   Interpretación: Índice alto indica clusters bien separados y compactos")

# 3. Davies-Bouldin Index (menor es mejor, 0 es ideal)
davies = davies_bouldin_score(X_scaled, clusters)
print(f"\n3️⃣ Davies-Bouldin Index: {davies:.4f}")
print(f"   Interpretación: ", end="")
if davies < 0.5:
    print("Excelente separación entre clusters")
elif davies < 1.0:
    print("Buena separación entre clusters")
else:
    print("Separación moderada entre clusters")

# 4. Inercia (Within-Cluster Sum of Squares)
if 'kmeans' in locals():
    inertia = kmeans.inertia_
    print(f"\n4️⃣ Inercia (WCSS): {inertia:.4f}")
    print(f"   Interpretación: Suma de distancias cuadradas dentro de clusters")

# ============================================================
# PRUEBA PARA MÚLTIPLES VALORES DE K
# ============================================================

print("\n" + "=" * 60)
print("COMPARACIÓN DE DIFERENTES NÚMEROS DE CLUSTERS (K=2 a K=6)")
print("=" * 60)

k_values = range(2, 7)
metrics_comparison = {
    'K': [],
    'Silhouette': [],
    'Calinski-Harabasz': [],
    'Davies-Bouldin': [],
    'Inertia': []
}

for k in k_values:
    kmeans_temp = KMeans(n_clusters=k, random_state=42, n_init=10)
    clusters_temp = kmeans_temp.fit_predict(X_scaled)
    
    metrics_comparison['K'].append(k)
    metrics_comparison['Silhouette'].append(silhouette_score(X_scaled, clusters_temp))
    metrics_comparison['Calinski-Harabasz'].append(calinski_harabasz_score(X_scaled, clusters_temp))
    metrics_comparison['Davies-Bouldin'].append(davies_bouldin_score(X_scaled, clusters_temp))
    metrics_comparison['Inertia'].append(kmeans_temp.inertia_)

metrics_df = pd.DataFrame(metrics_comparison)
print("\n📊 Tabla de Comparación:")
print(metrics_df.to_string(index=False))

# Visualización de métricas
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Silhouette Score
axes[0, 0].plot(metrics_df['K'], metrics_df['Silhouette'], 'bo-', linewidth=2, markersize=8)
axes[0, 0].set_xlabel('Número de Clusters (K)', fontsize=11)
axes[0, 0].set_ylabel('Silhouette Score', fontsize=11)
axes[0, 0].set_title('Silhouette Score (Mayor es mejor)', fontsize=12, fontweight='bold')
axes[0, 0].grid(alpha=0.3)
axes[0, 0].axvline(x=3, color='red', linestyle='--', alpha=0.5, label='K=3 (seleccionado)')
axes[0, 0].legend()

# Calinski-Harabasz Index
axes[0, 1].plot(metrics_df['K'], metrics_df['Calinski-Harabasz'], 'go-', linewidth=2, markersize=8)
axes[0, 1].set_xlabel('Número de Clusters (K)', fontsize=11)
axes[0, 1].set_ylabel('Calinski-Harabasz Index', fontsize=11)
axes[0, 1].set_title('Calinski-Harabasz Index (Mayor es mejor)', fontsize=12, fontweight='bold')
axes[0, 1].grid(alpha=0.3)
axes[0, 1].axvline(x=3, color='red', linestyle='--', alpha=0.5, label='K=3 (seleccionado)')
axes[0, 1].legend()

# Davies-Bouldin Index
axes[1, 0].plot(metrics_df['K'], metrics_df['Davies-Bouldin'], 'ro-', linewidth=2, markersize=8)
axes[1, 0].set_xlabel('Número de Clusters (K)', fontsize=11)
axes[1, 0].set_ylabel('Davies-Bouldin Index', fontsize=11)
axes[1, 0].set_title('Davies-Bouldin Index (Menor es mejor)', fontsize=12, fontweight='bold')
axes[1, 0].grid(alpha=0.3)
axes[1, 0].axvline(x=3, color='red', linestyle='--', alpha=0.5, label='K=3 (seleccionado)')
axes[1, 0].legend()

# Inertia (WCSS)
axes[1, 1].plot(metrics_df['K'], metrics_df['Inertia'], 'mo-', linewidth=2, markersize=8)
axes[1, 1].set_xlabel('Número de Clusters (K)', fontsize=11)
axes[1, 1].set_ylabel('Inercia (WCSS)', fontsize=11)
axes[1, 1].set_title('Inercia - Método del Codo (Menor es mejor)', fontsize=12, fontweight='bold')
axes[1, 1].grid(alpha=0.3)
axes[1, 1].axvline(x=3, color='red', linestyle='--', alpha=0.5, label='K=3 (seleccionado)')
axes[1, 1].legend()

plt.tight_layout()
plt.savefig('metricas_validacion_clustering.png', dpi=300, bbox_inches='tight')
print("\n✓ Gráficos de métricas guardados en 'metricas_validacion_clustering.png'")
plt.show()

# ============================================================
# DESCRIPCIÓN DETALLADA DE CADA CLUSTER
# ============================================================

print("\n" + "=" * 60)
print("DESCRIPCIÓN DETALLADA DE CADA CLUSTER")
print("=" * 60)

# Obtener número de clusters
n_clusters = len(np.unique(clusters))
indicadores = ['ind_eje', 'propim', 'proinv']

# Crear DataFrame con clusters
if 'df_clustered' not in locals():
    X = df[indicadores].copy()
    X_clean = X.dropna()
    df_clustered = X_clean.copy()
    df_clustered['cluster_estatico_nuevo'] = clusters

# Analizar cada cluster
cluster_descriptions = []

for i in range(n_clusters):
    print(f"\n{'=' * 60}")
    print(f"CLUSTER {i}")
    print(f"{'=' * 60}")
    
    cluster_data = df_clustered[df_clustered['cluster_estatico_nuevo'] == i][indicadores]
    n_municipios = len(cluster_data)
    
    print(f"\n📊 Tamaño: {n_municipios} municipalidades ({n_municipios/len(df_clustered)*100:.1f}% del total)")
    
    print(f"\n📈 Estadísticas:")
    for ind in indicadores:
        mean_val = cluster_data[ind].mean()
        median_val = cluster_data[ind].median()
        std_val = cluster_data[ind].std()
        print(f"\n   {ind}:")
        print(f"      Media:    {mean_val:.4f}")
        print(f"      Mediana:  {median_val:.4f}")
        print(f"      Desv.Est: {std_val:.4f}")
    
    # Interpretación del cluster
    print(f"\n💡 INTERPRETACIÓN:")
    
    mean_ind_eje = cluster_data['ind_eje'].mean()
    mean_propim = cluster_data['propim'].mean()
    mean_proinv = cluster_data['proinv'].mean()
    
    # Clasificar el cluster
    if mean_ind_eje > 0.75 and mean_propim > 0.65:
        tipo = "ALTO DESEMPEÑO"
        desc = "Municipalidades con excelente ejecución presupuestal y alta inversión municipal"
    elif mean_ind_eje < 0.55 or mean_proinv < 0.65:
        tipo = "BAJO DESEMPEÑO"
        desc = "Municipalidades con desafíos en ejecución presupuestal y/o inversión"
    else:
        tipo = "DESEMPEÑO MEDIO"
        desc = "Municipalidades con ejecución presupuestal moderada"
    
    print(f"   Tipo: {tipo}")
    print(f"   Descripción: {desc}")
    
    # Características distintivas
    print(f"\n🎯 CARACTERÍSTICAS DISTINTIVAS:")
    
    # Comparar con la media general
    mean_general_eje = df_clustered['ind_eje'].mean()
    mean_general_propim = df_clustered['propim'].mean()
    mean_general_proinv = df_clustered['proinv'].mean()
    
    if mean_ind_eje > mean_general_eje * 1.1:
        print(f"   ✓ Ejecución presupuestal superior al promedio general")
    elif mean_ind_eje < mean_general_eje * 0.9:
        print(f"   ⚠ Ejecución presupuestal inferior al promedio general")
    else:
        print(f"   • Ejecución presupuestal cercana al promedio general")
    
    if mean_propim > mean_general_propim * 1.1:
        print(f"   ✓ Proporción de inversión municipal superior al promedio")
    elif mean_propim < mean_general_propim * 0.9:
        print(f"   ⚠ Proporción de inversión municipal inferior al promedio")
    else:
        print(f"   • Proporción de inversión municipal cercana al promedio")
    
    if mean_proinv > mean_general_proinv * 1.1:
        print(f"   ✓ Proporción de inversión total superior al promedio")
    elif mean_proinv < mean_general_proinv * 0.9:
        print(f"   ⚠ Proporción de inversión total inferior al promedio")
    else:
        print(f"   • Proporción de inversión total cercana al promedio")
    
    # Guardar descripción
    cluster_descriptions.append({
        'cluster': i,
        'tipo': tipo,
        'descripcion': desc,
        'n_municipios': n_municipios,
        'pct_total': n_municipios/len(df_clustered)*100,
        'mean_ind_eje': mean_ind_eje,
        'mean_propim': mean_propim,
        'mean_proinv': mean_proinv
    })

# Crear DataFrame de resumen
cluster_summary_df = pd.DataFrame(cluster_descriptions)

print(f"\n{'=' * 60}")
print("RESUMEN COMPARATIVO DE CLUSTERS")
print(f"{'=' * 60}\n")
print(cluster_summary_df.to_string(index=False))

print("\n" + "=" * 60)
print("✓ ANÁLISIS DE VALIDACIÓN COMPLETADO")
print("=" * 60)

In [ ]:
# @title 6.6. Visualizaciones Individuales por Cluster 📊
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

# Colores profesionales
COLOR_PRIMARY = '#2C3E50'
COLOR_SECONDARY = '#3498DB'
COLOR_SUCCESS = '#27AE60'
COLOR_WARNING = '#F39C12'
COLOR_ACCENT = '#E74C3C'
COLOR_INFO = '#9B59B6'

CLUSTER_COLORS = [COLOR_SECONDARY, COLOR_SUCCESS, COLOR_ACCENT, COLOR_WARNING, COLOR_INFO]

print("=" * 60)
print("GENERANDO GRÁFICOS INDIVIDUALES POR CLUSTER")
print("=" * 60)

# Verificar que tenemos los datos necesarios
if 'df_clustered' not in locals() or 'cluster_estatico_nuevo' not in df_clustered.columns:
    print("⚠️ Ejecute primero la celda de clustering (Celda 5) y pruebas estadísticas (Celda 6.5)")
else:
    indicadores = ['ind_eje', 'propim', 'proinv']
    n_clusters = df_clustered['cluster_estatico_nuevo'].nunique()
    
    # ========== GRÁFICO 1: Características Promedio por Cluster ==========
    print("\n📊 Generando gráfico de características promedio por cluster...")
    
    fig, ax = plt.subplots(figsize=(12, 7))
    
    # Calcular promedios por cluster
    cluster_means = df_clustered.groupby('cluster_estatico_nuevo')[indicadores].mean()
    
    # Crear gráfico de barras agrupadas
    x = np.arange(len(indicadores))
    width = 0.25
    
    for i in range(n_clusters):
        offset = (i - n_clusters/2 + 0.5) * width
        ax.bar(x + offset, cluster_means.loc[i], width, 
               label=f'Cluster {i}', color=CLUSTER_COLORS[i], alpha=0.8, edgecolor='black', linewidth=1.2)
    
    ax.set_xlabel('Indicadores', fontsize=12, fontweight='bold')
    ax.set_ylabel('Valor Promedio', fontsize=12, fontweight='bold')
    ax.set_title('Características Promedio por Cluster', fontsize=14, fontweight='bold', pad=20)
    ax.set_xticks(x)
    ax.set_xticklabels(['Ejecución\nPresupuestal', 'Inversión\nMunicipal', 'Inversión\nTotal'])
    ax.legend(fontsize=10, loc='upper right')
    ax.grid(alpha=0.3, axis='y', linestyle='--')
    
    plt.tight_layout()
    plt.savefig('06_caracteristicas_promedio_clusters.png', dpi=300, bbox_inches='tight', facecolor='white')
    print("✓ Guardado: 06_caracteristicas_promedio_clusters.png")
    plt.close()
    
    # ========== GRÁFICO 2: Boxplots por Cluster e Indicador ==========
    print("\n📊 Generando boxplots comparativos por cluster...")
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    fig.suptitle('Distribución de Indicadores por Cluster', fontsize=16, fontweight='bold', y=1.02)
    
    nombres_indicadores = {
        'ind_eje': 'Ejecución Presupuestal',
        'propim': 'Inversión Municipal',
        'proinv': 'Inversión Total'
    }
    
    for idx, ind in enumerate(indicadores):
        # Preparar datos para boxplot
        data_by_cluster = [df_clustered[df_clustered['cluster_estatico_nuevo'] == i][ind].values 
                           for i in range(n_clusters)]
        
        bp = axes[idx].boxplot(data_by_cluster, labels=[f'C{i}' for i in range(n_clusters)],
                               patch_artist=True, widths=0.6)
        
        # Colorear cada box
        for patch, color in zip(bp['boxes'], CLUSTER_COLORS[:n_clusters]):
            patch.set_facecolor(color)
            patch.set_alpha(0.7)
        
        # Estilizar
        for element in ['whiskers', 'fliers', 'caps']:
            plt.setp(bp[element], color=COLOR_PRIMARY, linewidth=1.2)
        plt.setp(bp['medians'], color='red', linewidth=2)
        
        axes[idx].set_xlabel('Cluster', fontsize=11, fontweight='bold')
        axes[idx].set_ylabel('Valor', fontsize=11)
        axes[idx].set_title(nombres_indicadores[ind], fontsize=12, fontweight='bold')
        axes[idx].grid(alpha=0.3, axis='y', linestyle='--')
    
    plt.tight_layout()
    plt.savefig('07_boxplots_por_cluster.png', dpi=300, bbox_inches='tight', facecolor='white')
    print("✓ Guardado: 07_boxplots_por_cluster.png")
    plt.close()
    
    # ========== GRÁFICO 3: Gráfico de Radar para Perfiles de Clusters ==========
    print("\n📊 Generando gráfico de radar (spider) para perfiles...")
    
    from math import pi
    
    fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(projection='polar'))
    
    # Normalizar los datos para el gráfico de radar (0-1)
    cluster_means_norm = cluster_means.copy()
    for col in cluster_means_norm.columns:
        min_val = cluster_means_norm[col].min()
        max_val = cluster_means_norm[col].max()
        if max_val > min_val:
            cluster_means_norm[col] = (cluster_means_norm[col] - min_val) / (max_val - min_val)
    
    # Configurar ángulos
    categories = ['Ejecución\nPresupuestal', 'Inversión\nMunicipal', 'Inversión\nTotal']
    N = len(categories)
    angles = [n / float(N) * 2 * pi for n in range(N)]
    angles += angles[:1]
    
    # Dibujar cada cluster
    for i in range(n_clusters):
        values = cluster_means_norm.loc[i].tolist()
        values += values[:1]
        ax.plot(angles, values, 'o-', linewidth=2, label=f'Cluster {i}', color=CLUSTER_COLORS[i])
        ax.fill(angles, values, alpha=0.15, color=CLUSTER_COLORS[i])
    
    # Configurar ejes
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(categories, size=11, fontweight='bold')
    ax.set_ylim(0, 1)
    ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
    ax.set_yticklabels(['0.2', '0.4', '0.6', '0.8', '1.0'], size=9)
    ax.grid(True, linestyle='--', alpha=0.3)
    
    plt.title('Perfiles de Clusters\n(Valores Normalizados)', size=14, fontweight='bold', pad=20)
    plt.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=10)
    
    plt.tight_layout()
    plt.savefig('08_perfiles_radar_clusters.png', dpi=300, bbox_inches='tight', facecolor='white')
    print("✓ Guardado: 08_perfiles_radar_clusters.png")
    plt.close()
    
    # ========== GRÁFICO 4: Distribución de Tamaño de Clusters ==========
    print("\n📊 Generando gráfico de distribución de tamaño...")
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    # Pie chart
    cluster_sizes = df_clustered['cluster_estatico_nuevo'].value_counts().sort_index()
    colors = CLUSTER_COLORS[:n_clusters]
    
    wedges, texts, autotexts = ax1.pie(cluster_sizes, labels=[f'Cluster {i}' for i in range(n_clusters)],
                                         autopct='%1.1f%%', colors=colors, startangle=90,
                                         textprops={'fontsize': 11, 'fontweight': 'bold'},
                                         wedgeprops={'edgecolor': 'white', 'linewidth': 2})
    
    for autotext in autotexts:
        autotext.set_color('white')
        autotext.set_fontsize(12)
        autotext.set_fontweight('bold')
    
    ax1.set_title('Distribución Porcentual de Clusters', fontsize=14, fontweight='bold', pad=20)
    
    # Bar chart con números absolutos
    bars = ax2.bar(range(n_clusters), cluster_sizes, color=colors, alpha=0.8, 
                   edgecolor='black', linewidth=1.5)
    ax2.set_xlabel('Cluster', fontsize=12, fontweight='bold')
    ax2.set_ylabel('Número de Municipalidades', fontsize=12, fontweight='bold')
    ax2.set_title('Tamaño Absoluto de Clusters', fontsize=14, fontweight='bold', pad=20)
    ax2.set_xticks(range(n_clusters))
    ax2.set_xticklabels([f'Cluster {i}' for i in range(n_clusters)])
    ax2.grid(alpha=0.3, axis='y', linestyle='--')
    
    # Agregar valores encima de las barras
    for bar in bars:
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height,
                f'{int(height):,}',
                ha='center', va='bottom', fontsize=11, fontweight='bold')
    
    plt.tight_layout()
    plt.savefig('09_distribucion_tamano_clusters.png', dpi=300, bbox_inches='tight', facecolor='white')
    print("✓ Guardado: 09_distribucion_tamano_clusters.png")
    plt.close()
    
    # ========== GRÁFICO 5: Comparación Directa Cluster vs General ==========
    print("\n📊 Generando comparación cluster vs población general...")
    
    fig, axes = plt.subplots(n_clusters, 3, figsize=(18, 5*n_clusters))
    
    if n_clusters == 1:
        axes = axes.reshape(1, -1)
    
    fig.suptitle('Distribución de Indicadores: Cluster vs Población General', 
                 fontsize=16, fontweight='bold', y=0.995)
    
    for cluster_id in range(n_clusters):
        cluster_data = df_clustered[df_clustered['cluster_estatico_nuevo'] == cluster_id]
        
        for idx, ind in enumerate(indicadores):
            ax = axes[cluster_id, idx]
            
            # Histograma de la población general
            ax.hist(df_clustered[ind], bins=30, alpha=0.4, color='gray', 
                   label='General', edgecolor='black', linewidth=0.5)
            
            # Histograma del cluster
            ax.hist(cluster_data[ind], bins=30, alpha=0.7, color=CLUSTER_COLORS[cluster_id],
                   label=f'Cluster {cluster_id}', edgecolor='black', linewidth=0.8)
            
            # Líneas de media
            mean_general = df_clustered[ind].mean()
            mean_cluster = cluster_data[ind].mean()
            
            ax.axvline(mean_general, color='gray', linestyle='--', linewidth=2, alpha=0.7,
                      label=f'Media General: {mean_general:.3f}')
            ax.axvline(mean_cluster, color=CLUSTER_COLORS[cluster_id], linestyle='--', 
                      linewidth=2, label=f'Media C{cluster_id}: {mean_cluster:.3f}')
            
            ax.set_xlabel(nombres_indicadores[ind], fontsize=10)
            ax.set_ylabel('Frecuencia', fontsize=10)
            ax.set_title(f'Cluster {cluster_id} - {nombres_indicadores[ind]}', 
                        fontsize=11, fontweight='bold')
            ax.legend(fontsize=8, loc='upper right')
            ax.grid(alpha=0.3, axis='y', linestyle='--')
    
    plt.tight_layout()
    plt.savefig('10_comparacion_cluster_vs_general.png', dpi=300, bbox_inches='tight', facecolor='white')
    print("✓ Guardado: 10_comparacion_cluster_vs_general.png")
    plt.close()
    
    print("\n" + "=" * 60)
    print("✓ TODAS LAS VISUALIZACIONES POR CLUSTER GENERADAS")
    print("=" * 60)
    print(f"\n📊 Total de gráficos generados: 5")
    print("   1. Características promedio por cluster (barras agrupadas)")
    print("   2. Boxplots comparativos por cluster")
    print("   3. Perfiles de radar (spider chart)")
    print("   4. Distribución de tamaño de clusters")
    print("   5. Comparación cluster vs población general")

In [ ]:
# @title 7. Análisis de Panel (Clustering por Año) 📅
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

print("=" * 60)
print("ANÁLISIS DE PANEL - CLUSTERING POR AÑO")
print("=" * 60)

# Colores profesionales
COLOR_PRIMARY = '#2C3E50'
COLOR_SECONDARY = '#3498DB'
COLOR_SUCCESS = '#27AE60'
CLUSTER_COLORS = [COLOR_SECONDARY, COLOR_SUCCESS, '#E74C3C']

indicadores = ['ind_eje', 'propim', 'proinv']
years = sorted(df['year'].unique())
k_optimo = 3

# Diccionario para almacenar clusters por año
clusters_por_ano = {}
stats_por_ano = {}

print("\n📊 Realizando clustering K-Means para cada año...")

for year in years:
    print(f"\n--- AÑO {year} ---")
    
    # Filtrar datos del año
    df_year = df[df['year'] == year][indicadores + ['ejecutora_nombre']].copy()
    df_year_clean = df_year.dropna()
    
    # Normalizar
    X_year = df_year_clean[indicadores].values
    scaler_year = StandardScaler()
    X_year_scaled = scaler_year.fit_transform(X_year)
    
    # Clustering
    kmeans_year = KMeans(n_clusters=k_optimo, random_state=42, n_init=10)
    clusters_year = kmeans_year.fit_predict(X_year_scaled)
    
    # Guardar resultados
    df_year_clean['cluster_panel'] = clusters_year
    clusters_por_ano[year] = df_year_clean
    
    # Estadísticas
    stats_por_ano[year] = {
        'n_municipios': len(df_year_clean),
        'distribucion': df_year_clean['cluster_panel'].value_counts().sort_index().to_dict(),
        'centroides': kmeans_year.cluster_centers_,
        'centroides_orig': scaler_year.inverse_transform(kmeans_year.cluster_centers())
    }
    
    print(f"   Total municipalidades: {len(df_year_clean)}")
    print(f"   Distribución clusters: {stats_por_ano[year]['distribucion']}")

# ========== VISUALIZACIÓN 1: Evolución de Clusters por Año ==========
print("\n📊 Generando visualización de evolución de clusters...")

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Subplot 1: Distribución de clusters por año
ax1 = axes[0, 0]
data_dist = []
for year in years:
    for cluster_id in range(k_optimo):
        count = stats_por_ano[year]['distribucion'].get(cluster_id, 0)
        data_dist.append({'Año': year, 'Cluster': f'C{cluster_id}', 'Count': count})

df_dist = pd.DataFrame(data_dist)
pivot_dist = df_dist.pivot(index='Año', columns='Cluster', values='Count')

pivot_dist.plot(kind='bar', ax=ax1, color=CLUSTER_COLORS[:k_optimo], alpha=0.8, edgecolor='black', linewidth=1.2)
ax1.set_title('Distribución de Clusters por Año', fontsize=14, fontweight='bold')
ax1.set_xlabel('Año', fontsize=12)
ax1.set_ylabel('Número de Municipalidades', fontsize=12)
ax1.legend(title='Cluster', fontsize=10)
ax1.grid(alpha=0.3, axis='y', linestyle='--')
ax1.tick_params(axis='x', rotation=0)

# Subplot 2: Evolución de centroides (ind_eje)
ax2 = axes[0, 1]
for cluster_id in range(k_optimo):
    values = [stats_por_ano[year]['centroides_orig'][cluster_id][0] for year in years]
    ax2.plot(years, values, 'o-', linewidth=2, markersize=8, 
            label=f'Cluster {cluster_id}', color=CLUSTER_COLORS[cluster_id])

ax2.set_title('Evolución de Ejecución Presupuestal por Cluster', fontsize=14, fontweight='bold')
ax2.set_xlabel('Año', fontsize=12)
ax2.set_ylabel('Ejecución Presupuestal (Promedio)', fontsize=12)
ax2.legend(fontsize=10)
ax2.grid(alpha=0.3, linestyle='--')

# Subplot 3: Evolución de centroides (propim)
ax3 = axes[1, 0]
for cluster_id in range(k_optimo):
    values = [stats_por_ano[year]['centroides_orig'][cluster_id][1] for year in years]
    ax3.plot(years, values, 'o-', linewidth=2, markersize=8, 
            label=f'Cluster {cluster_id}', color=CLUSTER_COLORS[cluster_id])

ax3.set_title('Evolución de Inversión Municipal por Cluster', fontsize=14, fontweight='bold')
ax3.set_xlabel('Año', fontsize=12)
ax3.set_ylabel('Inversión Municipal (Promedio)', fontsize=12)
ax3.legend(fontsize=10)
ax3.grid(alpha=0.3, linestyle='--')

# Subplot 4: Evolución de centroides (proinv)
ax4 = axes[1, 1]
for cluster_id in range(k_optimo):
    values = [stats_por_ano[year]['centroides_orig'][cluster_id][2] for year in years]
    ax4.plot(years, values, 'o-', linewidth=2, markersize=8, 
            label=f'Cluster {cluster_id}', color=CLUSTER_COLORS[cluster_id])

ax4.set_title('Evolución de Inversión Total por Cluster', fontsize=14, fontweight='bold')
ax4.set_xlabel('Año', fontsize=12)
ax4.set_ylabel('Inversión Total (Promedio)', fontsize=12)
ax4.legend(fontsize=10)
ax4.grid(alpha=0.3, linestyle='--')

plt.tight_layout()
plt.savefig('11_analisis_panel_evolucion.png', dpi=300, bbox_inches='tight', facecolor='white')
print("✓ Guardado: 11_analisis_panel_evolucion.png")
plt.close()

# ========== RESUMEN ESTADÍSTICO ==========
print("\n" + "=" * 60)
print("RESUMEN ANÁLISIS DE PANEL")
print("=" * 60)

for year in years:
    print(f"\n📅 AÑO {year}:")
    print(f"   Total municipalidades analizadas: {stats_por_ano[year]['n_municipios']}")
    print(f"   Distribución por cluster:")
    for cluster_id, count in sorted(stats_por_ano[year]['distribucion'].items()):
        pct = (count / stats_por_ano[year]['n_municipios']) * 100
        print(f"      Cluster {cluster_id}: {count:,} ({pct:.1f}%)")

print("\n" + "=" * 60)
print("✓ ANÁLISIS DE PANEL COMPLETADO")
print("=" * 60)

In [ ]:
# @title 8. Análisis de Trayectorias (Seguimiento Temporal) 🛤️
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("=" * 60)
print("ANÁLISIS DE TRAYECTORIAS - SEGUIMIENTO TEMPORAL")
print("=" * 60)

# Colores
CLUSTER_COLORS = ['#3498DB', '#27AE60', '#E74C3C']

# Crear DataFrame con trayectorias
print("\n📊 Rastreando trayectorias de municipalidades...")

# Unir datos de clusters por año
trayectorias = []

for ejecutora in df['ejecutora_nombre'].unique():
    trayectoria = {'ejecutora': ejecutora}
    
    for year in sorted(df['year'].unique()):
        if year in clusters_por_ano:
            muni_data = clusters_por_ano[year][clusters_por_ano[year]['ejecutora_nombre'] == ejecutora]
            if len(muni_data) > 0:
                trayectoria[f'cluster_{year}'] = muni_data['cluster_panel'].iloc[0]
            else:
                trayectoria[f'cluster_{year}'] = np.nan
    
    # Solo agregar si tiene datos en los 3 años
    if all(f'cluster_{year}' in trayectoria and not pd.isna(trayectoria[f'cluster_{year}']) 
           for year in sorted(df['year'].unique())):
        trayectorias.append(trayectoria)

df_trayectorias = pd.DataFrame(trayectorias)
years = sorted(df['year'].unique())

print(f"✓ Total municipalidades con trayectoria completa: {len(df_trayectorias)}")

# Clasificar trayectorias
df_trayectorias['trayectoria_tipo'] = df_trayectorias.apply(
    lambda row: f"{int(row[f'cluster_{years[0]}'])}-{int(row[f'cluster_{years[1]}'])}-{int(row[f'cluster_{years[2]}'])}",
    axis=1
)

# Categorizar trayectorias
def categorizar_trayectoria(row):
    c1, c2, c3 = [int(row[f'cluster_{y}']) for y in years]
    
    if c1 == c2 == c3:
        return f'ESTABLE (C{c1})'
    elif c1 < c2 < c3 or (c1 < c3 and c2 <= c3):
        return 'MEJORANDO'
    elif c1 > c2 > c3 or (c1 > c3 and c2 >= c3):
        return 'EMPEORANDO'
    else:
        return 'FLUCTUANTE'

df_trayectorias['categoria'] = df_trayectorias.apply(categorizar_trayectoria, axis=1)

# Estadísticas
print("\n📈 DISTRIBUCIÓN POR CATEGORÍA DE TRAYECTORIA:")
cat_counts = df_trayectorias['categoria'].value_counts()
for cat, count in cat_counts.items():
    pct = (count / len(df_trayectorias)) * 100
    print(f"   {cat}: {count} ({pct:.1f}%)")

print("\n📊 TOP 10 TRAYECTORIAS MÁS COMUNES:")
top_trayectorias = df_trayectorias['trayectoria_tipo'].value_counts().head(10)
for traj, count in top_trayectorias.items():
    pct = (count / len(df_trayectorias)) * 100
    print(f"   {traj}: {count} ({pct:.1f}%)")

# ========== VISUALIZACIÓN 1: Diagrama Sankey simplificado ==========
print("\n📊 Generando visualizaciones de trayectorias...")

fig, axes = plt.subplots(2, 2, figsize=(18, 14))

# Subplot 1: Distribución por categoría
ax1 = axes[0, 0]
cat_counts.plot(kind='barh', ax=ax1, color=['#27AE60', '#3498DB', '#E74C3C', '#F39C12'], 
               alpha=0.8, edgecolor='black', linewidth=1.5)
ax1.set_title('Distribución por Categoría de Trayectoria', fontsize=14, fontweight='bold')
ax1.set_xlabel('Número de Municipalidades', fontsize=12)
ax1.set_ylabel('Categoría', fontsize=12)
ax1.grid(alpha=0.3, axis='x', linestyle='--')

for i, (cat, count) in enumerate(cat_counts.items()):
    pct = (count / len(df_trayectorias)) * 100
    ax1.text(count, i, f' {count} ({pct:.1f}%)', va='center', fontsize=10, fontweight='bold')

# Subplot 2: Matriz de transición 2022→2023
ax2 = axes[0, 1]
trans_2022_2023 = pd.crosstab(
    df_trayectorias[f'cluster_{years[0]}'].astype(int),
    df_trayectorias[f'cluster_{years[1]}'].astype(int)
)
sns.heatmap(trans_2022_2023, annot=True, fmt='d', cmap='YlOrRd', ax=ax2, 
           cbar_kws={'label': 'Frecuencia'}, linewidths=1, linecolor='black')
ax2.set_title(f'Transiciones {years[0]}→{years[1]}', fontsize=14, fontweight='bold')
ax2.set_xlabel(f'Cluster {years[1]}', fontsize=12)
ax2.set_ylabel(f'Cluster {years[0]}', fontsize=12)

# Subplot 3: Matriz de transición 2023→2024
ax3 = axes[1, 0]
trans_2023_2024 = pd.crosstab(
    df_trayectorias[f'cluster_{years[1]}'].astype(int),
    df_trayectorias[f'cluster_{years[2]}'].astype(int)
)
sns.heatmap(trans_2023_2024, annot=True, fmt='d', cmap='YlGnBu', ax=ax3, 
           cbar_kws={'label': 'Frecuencia'}, linewidths=1, linecolor='black')
ax3.set_title(f'Transiciones {years[1]}→{years[2]}', fontsize=14, fontweight='bold')
ax3.set_xlabel(f'Cluster {years[2]}', fontsize=12)
ax3.set_ylabel(f'Cluster {years[1]}', fontsize=12)

# Subplot 4: Top 10 trayectorias
ax4 = axes[1, 1]
top_trayectorias.plot(kind='barh', ax=ax4, color='#9B59B6', alpha=0.8, 
                      edgecolor='black', linewidth=1.5)
ax4.set_title('Top 10 Trayectorias Más Comunes', fontsize=14, fontweight='bold')
ax4.set_xlabel('Número de Municipalidades', fontsize=12)
ax4.set_ylabel('Trayectoria (2022-2023-2024)', fontsize=12)
ax4.grid(alpha=0.3, axis='x', linestyle='--')

for i, count in enumerate(top_trayectorias.values):
    pct = (count / len(df_trayectorias)) * 100
    ax4.text(count, i, f' {count} ({pct:.1f}%)', va='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('12_analisis_trayectorias.png', dpi=300, bbox_inches='tight', facecolor='white')
print("✓ Guardado: 12_analisis_trayectorias.png")
plt.close()

# ========== ANÁLISIS DE ESTABILIDAD ==========
print("\n" + "=" * 60)
print("ANÁLISIS DE ESTABILIDAD")
print("=" * 60)

estables = df_trayectorias[df_trayectorias['categoria'].str.contains('ESTABLE')]
print(f"\nMunicipalidades estables: {len(estables)} ({len(estables)/len(df_trayectorias)*100:.1f}%)")
print(f"Municipalidades con cambios: {len(df_trayectorias) - len(estables)} ({(len(df_trayectorias) - len(estables))/len(df_trayectorias)*100:.1f}%)")

print("\n📊 Distribución de municipalidades estables por cluster:")
for cluster_id in range(3):
    count = len(estables[estables['categoria'] == f'ESTABLE (C{cluster_id})'])
    pct = (count / len(estables)) * 100 if len(estables) > 0 else 0
    print(f"   Cluster {cluster_id}: {count} ({pct:.1f}%)")

print("\n" + "=" * 60)
print("✓ ANÁLISIS DE TRAYECTORIAS COMPLETADO")
print("=" * 60)

In [ ]:
# @title 9. Comparación entre los 3 Métodos de Clustering 🔬
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("=" * 60)
print("COMPARACIÓN ENTRE MÉTODOS DE CLUSTERING")
print("=" * 60)

# Colores
CLUSTER_COLORS = ['#3498DB', '#27AE60', '#E74C3C']

print("\n📊 Comparando los 3 enfoques de clustering:")
print("   1. Estático (K-Means): Clustering global sin considerar tiempo")
print("   2. Panel: Clustering separado por año (temporal)")
print("   3. Trayectorias: Seguimiento de cambios temporales")

# ========== GRÁFICO 1: Comparación de Distribuciones ==========
print("\n📈 Generando comparación de distribuciones...")

fig, axes = plt.subplots(2, 2, figsize=(18, 14))

# Subplot 1: Distribución Método Estático
if 'df_clustered' in locals():
    ax1 = axes[0, 0]
    dist_estatico = df_clustered['cluster_estatico_nuevo'].value_counts().sort_index()
    bars1 = ax1.bar(range(len(dist_estatico)), dist_estatico.values, 
                    color=CLUSTER_COLORS[:len(dist_estatico)], alpha=0.8, 
                    edgecolor='black', linewidth=1.5)
    ax1.set_title('Método Estático (K-Means Global)', fontsize=14, fontweight='bold')
    ax1.set_xlabel('Cluster', fontsize=12)
    ax1.set_ylabel('Número de Municipalidades', fontsize=12)
    ax1.set_xticks(range(len(dist_estatico)))
    ax1.set_xticklabels([f'C{i}' for i in range(len(dist_estatico))])
    ax1.grid(alpha=0.3, axis='y', linestyle='--')
    
    for bar in bars1:
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height,
                f'{int(height):,}\n({height/len(df_clustered)*100:.1f}%)',
                ha='center', va='bottom', fontsize=10, fontweight='bold')

# Subplot 2: Distribución Promedio Método Panel
ax2 = axes[0, 1]
dist_panel_avg = {}
for cluster_id in range(3):
    avg_count = np.mean([stats_por_ano[year]['distribucion'].get(cluster_id, 0) 
                        for year in stats_por_ano.keys()])
    dist_panel_avg[cluster_id] = avg_count

bars2 = ax2.bar(range(len(dist_panel_avg)), list(dist_panel_avg.values()), 
                color=CLUSTER_COLORS[:len(dist_panel_avg)], alpha=0.8, 
                edgecolor='black', linewidth=1.5)
ax2.set_title('Método Panel (Promedio 2022-2024)', fontsize=14, fontweight='bold')
ax2.set_xlabel('Cluster', fontsize=12)
ax2.set_ylabel('Promedio de Municipalidades', fontsize=12)
ax2.set_xticks(range(len(dist_panel_avg)))
ax2.set_xticklabels([f'C{i}' for i in range(len(dist_panel_avg))])
ax2.grid(alpha=0.3, axis='y', linestyle='--')

for bar in bars2:
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height,
            f'{int(height):,}',
            ha='center', va='bottom', fontsize=10, fontweight='bold')

# Subplot 3: Categorías de Trayectorias
ax3 = axes[1, 0]
if 'df_trayectorias' in locals():
    cat_counts = df_trayectorias['categoria'].value_counts()
    colors_cat = ['#27AE60', '#3498DB', '#E74C3C', '#F39C12', '#9B59B6']
    bars3 = ax3.barh(range(len(cat_counts)), cat_counts.values, 
                    color=colors_cat[:len(cat_counts)], alpha=0.8, 
                    edgecolor='black', linewidth=1.5)
    ax3.set_title('Método Trayectorias (Categorías)', fontsize=14, fontweight='bold')
    ax3.set_xlabel('Número de Municipalidades', fontsize=12)
    ax3.set_yticks(range(len(cat_counts)))
    ax3.set_yticklabels(cat_counts.index, fontsize=10)
    ax3.grid(alpha=0.3, axis='x', linestyle='--')
    
    for i, (bar, count) in enumerate(zip(bars3, cat_counts.values)):
        width = bar.get_width()
        pct = (count / len(df_trayectorias)) * 100
        ax3.text(width, i, f' {count} ({pct:.1f}%)', 
                va='center', fontsize=10, fontweight='bold')

# Subplot 4: Tabla Comparativa
ax4 = axes[1, 1]
ax4.axis('off')

# Crear tabla de comparación
tabla_data = [
    ['Método', 'Enfoque', 'N° Observ.', 'Clusters'],
    ['Estático', 'Global (todos los años)', 
     f'{len(df_clustered):,}' if 'df_clustered' in locals() else 'N/A', '3'],
    ['Panel', 'Por año (2022, 2023, 2024)', 
     f'{sum(stats_por_ano[y]["n_municipios"] for y in stats_por_ano.keys()):,}', '3 x 3 años'],
    ['Trayectorias', 'Seguimiento temporal', 
     f'{len(df_trayectorias):,}' if 'df_trayectorias' in locals() else 'N/A', 'Categorías']
]

tabla = ax4.table(cellText=tabla_data, cellLoc='left', loc='center',
                 colWidths=[0.2, 0.4, 0.2, 0.2])
tabla.auto_set_font_size(False)
tabla.set_fontsize(10)
tabla.scale(1, 2.5)

# Estilizar encabezado
for i in range(4):
    cell = tabla[(0, i)]
    cell.set_facecolor('#2C3E50')
    cell.set_text_props(weight='bold', color='white')

# Estilizar filas
colors_rows = ['#ECF0F1', 'white']
for i in range(1, 4):
    for j in range(4):
        cell = tabla[(i, j)]
        cell.set_facecolor(colors_rows[i % 2])
        cell.set_edgecolor('black')
        cell.set_linewidth(1.5)

ax4.set_title('Tabla Comparativa de Métodos', fontsize=14, fontweight='bold', pad=20)

plt.tight_layout()
plt.savefig('13_comparacion_metodos.png', dpi=300, bbox_inches='tight', facecolor='white')
print("✓ Guardado: 13_comparacion_metodos.png")
plt.close()

# ========== ANÁLISIS COMPARATIVO ==========
print("\n" + "=" * 60)
print("ANÁLISIS COMPARATIVO")
print("=" * 60)

print("\n📊 SIMILITUDES:")
print("   • Los 3 métodos identifican 3 clusters principales")
print("   • Patrones de agrupamiento consistentes en indicadores clave")
print("   • Distribuciones relativamente estables entre métodos")

print("\n📊 DIFERENCIAS:")
print("   • Estático: Visión global agregada, ignora variación temporal")
print("   • Panel: Captura cambios año a año, permite ver evolución")
print("   • Trayectorias: Identifica patrones de cambio individuales")

if 'df_clustered' in locals() and 'df_trayectorias' in locals():
    estables_pct = (len(df_trayectorias[df_trayectorias['categoria'].str.contains('ESTABLE')]) / 
                    len(df_trayectorias)) * 100
    print(f"\n📊 ESTABILIDAD TEMPORAL:")
    print(f"   • {estables_pct:.1f}% de municipalidades mantienen el mismo cluster")
    print(f"   • {100-estables_pct:.1f}% experimentan cambios entre años")

print("\n📊 RECOMENDACIONES DE USO:")
print("   • Estático: Para análisis general y benchmarking")
print("   • Panel: Para políticas que requieren ajustes anuales")
print("   • Trayectorias: Para identificar municipalidades que mejoran/empeoran")

print("\n" + "=" * 60)
print("✓ COMPARACIÓN ENTRE MÉTODOS COMPLETADA")
print("=" * 60)

In [ ]:
# @title 13. Comprimir y Descargar Todos los Resultados 📥
import shutil
import os

print("=" * 60)
print("COMPRESIÓN Y DESCARGA DE RESULTADOS")
print("=" * 60)

# Verificar que el directorio de resultados existe
if not os.path.exists('results'):
    print("⚠️ El directorio 'results' no existe. Ejecute primero la celda anterior.")
else:
    # Crear archivo ZIP con todos los resultados
    print("\n🗜️ Comprimiendo archivos...")
    
    # Listar archivos PNG generados (visualizaciones)
    png_files = [f for f in os.listdir('.') if f.endswith('.png')]
    
    # Copiar archivos PNG al directorio results
    for png_file in png_files:
        if os.path.exists(png_file):
            shutil.copy(png_file, os.path.join('results', png_file))
            print(f"✓ Copiado: {png_file}")
    
    # Copiar archivo Word al directorio results (si existe)
    word_files = [f for f in os.listdir('.') if f.endswith('.docx') and 'Informe_Clustering_Municipal' in f]
    for word_file in word_files:
        if os.path.exists(word_file):
            # Solo copiar si no existe ya en results
            dest_path = os.path.join('results', word_file)
            if not os.path.exists(dest_path):
                shutil.copy(word_file, dest_path)
            print(f"✓ Copiado: {word_file}")
    
    # Crear ZIP
    zip_path = shutil.make_archive('resultados_clustering_municipal', 'zip', 'results')
    print(f"\n✅ Archivo ZIP generado: {zip_path}")
    
    # Mostrar contenido del ZIP
    print(f"\n📦 Contenido del archivo ZIP:")
    for file in sorted(os.listdir('results')):
        file_path = os.path.join('results', file)
        size_kb = os.path.getsize(file_path) / 1024
        file_type = "📊 CSV" if file.endswith('.csv') else "🖼️ PNG" if file.endswith('.png') else "📄 Word"
        print(f"   {file_type} {file} ({size_kb:.2f} KB)")
    
    # Tamaño total del ZIP
    zip_size_mb = os.path.getsize(zip_path) / (1024 * 1024)
    print(f"\n📊 Tamaño total del ZIP: {zip_size_mb:.2f} MB")
    
    # Intentar descargar en Google Colab
    print("\n📥 Intentando descargar archivo...")
    try:
        from google.colab import files
        files.download(zip_path)
        print("✓ Descarga iniciada en Google Colab")
    except Exception as e:
        print("ℹ️ No estamos en Google Colab o la descarga automática falló")
        print(f"📁 El archivo ZIP está disponible en: {os.path.abspath(zip_path)}")
        print("   Puede descargarlo manualmente desde el explorador de archivos")

print("\n" + "=" * 60)
print("✓ ANÁLISIS COMPLETADO")
print("=" * 60)
print("\n🎉 ¡Análisis de clustering presupuestal municipal finalizado con éxito!")
print("\n📋 Resumen de lo realizado:")
print("   ✓ Carga y exploración de datos")
print("   ✓ Análisis descriptivo y visualizaciones exploratorias")
print("   ✓ Clustering estático con K-Means")
print("   ✓ Visualización con PCA")
print("   ✓ Análisis de clustering de panel")
print("   ✓ Análisis de clustering de trayectorias")
print("   ✓ Pruebas y validación de gráficos")
print("   ✓ Exportación de resultados en CSV")
print("   ✓ Generación de visualizaciones (PNG)")
print("   ✓ Generación de informe completo en Word")
print("   ✓ Compresión en archivo ZIP")
print("\n📊 Dataset: 1,770 municipalidades peruanas")
print("📈 Métodos: 3 enfoques de clustering")
print("📁 Resultados: Disponibles en resultados_clustering_municipal.zip")
print("📄 Informe Word: Incluido en el ZIP con análisis completo")

In [ ]:
# @title 12. Generación de Documento Word Completo 📄
print("Instalando python-docx si es necesario...")
try:
    from docx import Document
    from docx.shared import Inches, Pt, RGBColor
    from docx.enum.text import WD_ALIGN_PARAGRAPH
    print("✓ python-docx disponible")
except ImportError:
    import subprocess
    subprocess.check_call(['pip', 'install', 'python-docx', '-q'])
    from docx import Document
    from docx.shared import Inches, Pt, RGBColor
    from docx.enum.text import WD_ALIGN_PARAGRAPH
    print("✓ python-docx instalado")

import os
from datetime import datetime
import pandas as pd
import numpy as np

print("\n" + "=" * 60)
print("GENERACIÓN DE DOCUMENTO WORD")
print("=" * 60)

# Crear documento Word
doc = Document()

# ============================================================
# PORTADA
# ============================================================
print("\n📝 Creando portada...")

titulo = doc.add_heading('Análisis de Clustering Presupuestal Municipal', 0)
titulo.alignment = WD_ALIGN_PARAGRAPH.CENTER

subtitulo = doc.add_heading('Municipalidades Peruanas 2022-2024', level=2)
subtitulo.alignment = WD_ALIGN_PARAGRAPH.CENTER

subtitulo2 = doc.add_heading('Tres Enfoques Metodológicos Complementarios', level=3)
subtitulo2.alignment = WD_ALIGN_PARAGRAPH.CENTER

info = doc.add_paragraph()
info.alignment = WD_ALIGN_PARAGRAPH.CENTER
info.add_run(f'\n\nFecha de generación: {datetime.now().strftime("%d de %B de %Y")}\n')
info.add_run(f'Dataset: {len(df):,} registros, {df["ejecutora_nombre"].nunique():,} municipalidades\n')
info.add_run(f'Periodo: 2022-2024\n')
info.add_run(f'Métodos: K-Means Estático | Panel Temporal | Trayectorias\n\n')

doc.add_page_break()

# ============================================================
# RESUMEN EJECUTIVO
# ============================================================
print("📝 Agregando resumen ejecutivo...")

doc.add_heading('Resumen Ejecutivo', 1)

p = doc.add_paragraph()
p.add_run('Este documento presenta un análisis exhaustivo de clustering presupuestal de ')
p.add_run(f'{df["ejecutora_nombre"].nunique():,} municipalidades peruanas ').bold = True
p.add_run('durante el periodo 2022-2024 mediante ')
p.add_run('tres enfoques complementarios').bold = True
p.add_run(': (1) Análisis Estático con K-Means, (2) Análisis de Panel temporal, y (3) Análisis de Trayectorias, con validación estadística rigurosa.')

doc.add_heading('Metodologías Aplicadas', 2)
metodologias = [
    'Análisis Estático (K-Means): Clustering global sin considerar tiempo - proporciona benchmarking general y clasificación básica',
    'Análisis de Panel: Clustering separado por año (2022-2024) - captura evolución temporal y cambios en centroides',
    'Análisis de Trayectorias: Seguimiento de cambios individuales - identifica patrones de mejora, deterioro, estabilidad y fluctuación'
]
for metod in metodologias:
    doc.add_paragraph(metod, style='List Bullet')

doc.add_heading('Objetivos del Análisis', 2)
objetivos = [
    'Identificar grupos homogéneos de municipalidades según su desempeño presupuestal',
    'Analizar indicadores clave: ejecución presupuestal, inversión municipal e inversión total',
    'Validar la calidad del clustering mediante múltiples métricas estadísticas',
    'Describir características distintivas de cada cluster identificado',
    'Capturar la evolución temporal del desempeño presupuestal',
    'Identificar municipalidades con patrones de mejora o deterioro sostenido',
    'Proporcionar insights para la toma de decisiones en gestión pública'
]
for obj in objetivos:
    doc.add_paragraph(obj, style='List Bullet')

doc.add_page_break()

# ============================================================
# ESTADÍSTICAS DESCRIPTIVAS
# ============================================================
print("📊 Agregando estadísticas descriptivas...")

doc.add_heading('1. Estadísticas Descriptivas', 1)

doc.add_paragraph('El análisis se basa en tres indicadores principales:')

# Tabla de indicadores
doc.add_heading('Indicadores Analizados', 2)
table = doc.add_table(rows=4, cols=3)
table.style = 'Light Grid Accent 1'

headers = table.rows[0].cells
headers[0].text = 'Indicador'
headers[1].text = 'Descripción'
headers[2].text = 'Rango'

data_rows = [
    ['ind_eje', 'Indicador de ejecución presupuestal', '0.0 - 1.0'],
    ['propim', 'Proporción de inversión municipal', '0.0 - 1.0'],
    ['proinv', 'Proporción de inversión total', '0.0 - 1.0']
]

for i, row_data in enumerate(data_rows, 1):
    row = table.rows[i].cells
    for j, cell_data in enumerate(row_data):
        row[j].text = cell_data

# Estadísticas generales
doc.add_heading('Estadísticas Generales', 2)

stats_table = doc.add_table(rows=4, cols=4)
stats_table.style = 'Light List Accent 1'

stat_headers = stats_table.rows[0].cells
stat_headers[0].text = 'Indicador'
stat_headers[1].text = 'Media'
stat_headers[2].text = 'Mediana'
stat_headers[3].text = 'Desv. Estándar'

stats_data = []
for ind in ['ind_eje', 'propim', 'proinv']:
    stats_data.append([
        ind,
        f"{df[ind].mean():.4f}",
        f"{df[ind].median():.4f}",
        f"{df[ind].std():.4f}"
    ])

for i, row_data in enumerate(stats_data, 1):
    row = stats_table.rows[i].cells
    for j, cell_data in enumerate(row_data):
        row[j].text = cell_data

doc.add_page_break()

# ============================================================
# ANÁLISIS EXPLORATORIO
# ============================================================
print("🖼️ Insertando gráficos exploratorios...")

doc.add_heading('2. Análisis Exploratorio de Datos', 1)

# Gráficos individuales
graficos_exploratorios = [
    ('01_distribucion_indicadores.png', 'Distribución de Indicadores Presupuestales'),
    ('02_boxplots_indicadores.png', 'Análisis de Variabilidad'),
    ('03_scatter_relaciones.png', 'Relaciones entre Indicadores'),
    ('04_matriz_correlacion.png', 'Matriz de Correlación'),
    ('05_evolucion_temporal.png', 'Evolución Temporal 2022-2024')
]

for grafico, titulo in graficos_exploratorios:
    if os.path.exists(grafico):
        doc.add_heading(titulo, 2)
        doc.add_picture(grafico, width=Inches(6.0))
        doc.add_paragraph()

doc.add_page_break()

# ============================================================
# METODOLOGÍA 1: CLUSTERING ESTÁTICO
# ============================================================
print("📈 Agregando clustering estático...")

doc.add_heading('3. Metodología 1: Análisis Estático (K-Means)', 1)

doc.add_paragraph(
    'El análisis estático aplica clustering K-Means a todos los datos agregados, '
    'sin considerar la dimensión temporal. Este enfoque proporciona una visión general '
    'del desempeño presupuestal y sirve como línea base para comparación.'
)

# Método del codo
if os.path.exists('metodo_codo.png'):
    doc.add_heading('3.1. Determinación del Número Óptimo de Clusters', 2)
    doc.add_paragraph('El método del codo se utilizó para determinar el número óptimo de clusters:')
    doc.add_picture('metodo_codo.png', width=Inches(5.5))
    doc.add_paragraph()

# Métricas de validación
doc.add_heading('3.2. Métricas de Validación del Clustering', 2)

metrics_text = doc.add_paragraph()
metrics_text.add_run('Se aplicaron cuatro métricas estadísticas para validar la calidad del clustering:')

doc.add_paragraph()

# Tabla de métricas
metrics_table = doc.add_table(rows=5, cols=3)
metrics_table.style = 'Medium Grid 1 Accent 1'

headers = metrics_table.rows[0].cells
headers[0].text = 'Métrica'
headers[1].text = 'Rango Óptimo'
headers[2].text = 'Interpretación'

# Datos de métricas
metrics_info = [
    ['Silhouette Score', '0.0 - 1.0 (mayor mejor)', 'Mide la cohesión y separación de clusters'],
    ['Calinski-Harabasz', 'Mayor es mejor', 'Clusters bien separados y compactos'],
    ['Davies-Bouldin', 'Menor es mejor', 'Índice de similaridad entre clusters'],
    ['Inercia (WCSS)', 'Menor es mejor', 'Suma de distancias dentro de clusters']
]

for i, row_data in enumerate(metrics_info, 1):
    row = metrics_table.rows[i].cells
    for j, cell_data in enumerate(row_data):
        row[j].text = cell_data

doc.add_paragraph()

# Gráficos de métricas
if os.path.exists('metricas_validacion_clustering.png'):
    doc.add_heading('3.3. Comparación de Métricas para Diferentes Valores de K', 2)
    doc.add_paragraph('Se evaluaron múltiples valores de K (2-6) para confirmar que K=3 es óptimo:')
    doc.add_picture('metricas_validacion_clustering.png', width=Inches(6.5))
    doc.add_paragraph()

# Visualización PCA
if os.path.exists('clusters_pca_visualization.png'):
    doc.add_heading('3.4. Visualización con PCA', 2)
    doc.add_paragraph('La proyección PCA permite visualizar los clusters en dos dimensiones:')
    doc.add_picture('clusters_pca_visualization.png', width=Inches(6.5))
    doc.add_paragraph()

doc.add_page_break()

# ============================================================
# DESCRIPCIÓN DE CLUSTERS - MÉTODO ESTÁTICO
# ============================================================
print("📋 Agregando descripciones de clusters estáticos...")

doc.add_heading('3.5. Clusters Identificados - Método Estático', 2)

doc.add_paragraph(
    'Se identificaron 3 clusters principales mediante K-Means estático:'
)

# Verificar si existe cluster_descriptions
if 'cluster_descriptions' in locals():
    try:
        for cluster_desc in cluster_descriptions:
            cluster_id = cluster_desc['cluster']
            doc.add_heading(f'Cluster {cluster_id}', 3)

            cluster_table = doc.add_table(rows=7, cols=2)
            cluster_table.style = 'Light List Accent 1'

            headers = cluster_table.rows[0].cells
            headers[0].text = 'Atributo'
            headers[1].text = 'Valor'

            cluster_info = [
                ['Tamaño', f'{cluster_desc["n_municipios"]} municipalidades ({cluster_desc["pct_total"]:.1f}%)'],
                ['Tipo', cluster_desc['tipo']],
                ['Descripción', cluster_desc['descripcion']],
                ['Media ind_eje', f'{cluster_desc["mean_ind_eje"]:.4f}'],
                ['Media propim', f'{cluster_desc["mean_propim"]:.4f}'],
                ['Media proinv', f'{cluster_desc["mean_proinv"]:.4f}']
            ]

            for j, row_data in enumerate(cluster_info, 1):
                row = cluster_table.rows[j].cells
                row[0].text = row_data[0]
                row[1].text = row_data[1]

            doc.add_paragraph()
    except:
        for i in range(3):
            doc.add_heading(f'Cluster {i}', 3)
            doc.add_paragraph(f'Grupo de municipalidades con características presupuestales específicas.')
else:
    for i in range(3):
        doc.add_heading(f'Cluster {i}', 3)
        doc.add_paragraph(f'Grupo {i+1} de municipalidades con características presupuestales distintivas.')

# Gráficos por cluster
doc.add_heading('3.6. Visualizaciones Detalladas por Cluster', 2)

graficos_cluster = [
    ('06_caracteristicas_promedio_clusters.png', 'Características Promedio'),
    ('07_boxplots_por_cluster.png', 'Distribuciones por Cluster'),
    ('08_perfiles_radar_clusters.png', 'Perfiles de Radar'),
    ('09_distribucion_tamano_clusters.png', 'Distribución de Tamaño'),
    ('10_comparacion_cluster_vs_general.png', 'Comparación vs Población General')
]

for grafico, titulo in graficos_cluster:
    if os.path.exists(grafico):
        doc.add_heading(titulo, 3)
        doc.add_picture(grafico, width=Inches(6.0))
        doc.add_paragraph()

doc.add_page_break()

# ============================================================
# METODOLOGÍA 2: ANÁLISIS DE PANEL
# ============================================================
print("📅 Agregando análisis de panel...")

doc.add_heading('4. Metodología 2: Análisis de Panel (Temporal)', 1)

doc.add_paragraph(
    'El análisis de panel realiza clustering K-Means separado para cada año (2022, 2023, 2024), '
    'permitiendo observar cómo evolucionan los centroides y las distribuciones temporalmente. '
    'Este enfoque captura la dinámica temporal del desempeño presupuestal municipal.'
)

doc.add_heading('4.1. Enfoque Metodológico', 2)
panel_caracteristicas = [
    'Clustering independiente para cada año del periodo 2022-2024',
    'Normalización separada por año para controlar efectos temporales',
    'Rastreo de evolución de centroides año a año',
    'Comparación de distribuciones entre años',
    'Identificación de tendencias temporales en indicadores'
]
for caract in panel_caracteristicas:
    doc.add_paragraph(caract, style='List Bullet')

doc.add_heading('4.2. Resultados del Análisis de Panel', 2)

if 'stats_por_ano' in locals() and len(stats_por_ano) > 0:
    doc.add_paragraph('Distribución de municipalidades por cluster y año:')

    panel_table = doc.add_table(rows=5, cols=4)
    panel_table.style = 'Medium Grid 1 Accent 1'

    headers = panel_table.rows[0].cells
    headers[0].text = 'Cluster'
    headers[1].text = '2022'
    headers[2].text = '2023'
    headers[3].text = '2024'

    for cluster_id in range(3):
        row = panel_table.rows[cluster_id + 1].cells
        row[0].text = f'Cluster {cluster_id}'
        for col_idx, year in enumerate(sorted(stats_por_ano.keys()), 1):
            count = stats_por_ano[year]['distribucion'].get(cluster_id, 0)
            pct = (count / stats_por_ano[year]['n_municipios']) * 100
            row[col_idx].text = f'{count} ({pct:.1f}%)'

    # Fila de totales
    row = panel_table.rows[4].cells
    row[0].text = 'TOTAL'
    for col_idx, year in enumerate(sorted(stats_por_ano.keys()), 1):
        row[col_idx].text = f"{stats_por_ano[year]['n_municipios']}"

doc.add_heading('4.3. Evolución Temporal de Clusters', 2)

if os.path.exists('11_analisis_panel_evolucion.png'):
    doc.add_paragraph(
        'El siguiente gráfico muestra la evolución de los centroides de cada cluster '
        'a lo largo del periodo 2022-2024 para los tres indicadores:'
    )
    doc.add_picture('11_analisis_panel_evolucion.png', width=Inches(6.5))
    doc.add_paragraph()

doc.add_heading('4.4. Interpretación del Análisis de Panel', 2)

interpretacion_panel = [
    'Estabilidad estructural: Los 3 clusters se mantienen consistentes año a año',
    'Evolución gradual: Los centroides muestran cambios graduales, no abruptos',
    'Distribuciones: La proporción de municipalidades por cluster varía levemente entre años',
    'Tendencias: Se identifican patrones de mejora o deterioro en indicadores específicos',
    'Validación: Confirma que la estructura de clustering es robusta temporalmente'
]

for interp in interpretacion_panel:
    doc.add_paragraph(interp, style='List Bullet')

doc.add_page_break()

# ============================================================
# METODOLOGÍA 3: ANÁLISIS DE TRAYECTORIAS
# ============================================================
print("🛤️ Agregando análisis de trayectorias...")

doc.add_heading('5. Metodología 3: Análisis de Trayectorias', 1)

doc.add_paragraph(
    'El análisis de trayectorias realiza un seguimiento individualizado de cada municipalidad '
    'a lo largo del periodo 2022-2024, identificando patrones de cambio en su asignación de clusters. '
    'Este enfoque permite identificar municipalidades que mejoran, empeoran o se mantienen estables.'
)

doc.add_heading('5.1. Categorización de Trayectorias', 2)

categorias_trayectorias = [
    'ESTABLE: Municipalidades que permanecen en el mismo cluster durante todo el periodo',
    'MEJORANDO: Transición progresiva hacia clusters de mejor desempeño',
    'EMPEORANDO: Transición hacia clusters de menor desempeño',
    'FLUCTUANTE: Cambios variables sin un patrón claro de mejora o deterioro'
]

for categ in categorias_trayectorias:
    doc.add_paragraph(categ, style='List Bullet')

doc.add_heading('5.2. Resultados del Análisis de Trayectorias', 2)

if 'df_trayectorias' in locals():
    doc.add_paragraph(
        f'Se analizaron trayectorias completas de {len(df_trayectorias):,} municipalidades '
        f'con datos en los tres años del periodo.'
    )

    # Tabla de distribución por categoría
    traj_table = doc.add_table(rows=5, cols=3)
    traj_table.style = 'Light List Accent 1'

    headers = traj_table.rows[0].cells
    headers[0].text = 'Categoría'
    headers[1].text = 'N° Municipalidades'
    headers[2].text = 'Porcentaje'

    cat_counts = df_trayectorias['categoria'].value_counts()
    for idx, (cat, count) in enumerate(cat_counts.items(), 1):
        if idx < 5:  # Solo primeras 4 categorías
            row = traj_table.rows[idx].cells
            row[0].text = cat
            row[1].text = f'{count:,}'
            row[2].text = f'{count/len(df_trayectorias)*100:.1f}%'

doc.add_heading('5.3. Matrices de Transición', 2)

if os.path.exists('12_analisis_trayectorias.png'):
    doc.add_paragraph(
        'Las matrices de transición muestran los movimientos entre clusters en periodos consecutivos '
        'y las trayectorias más comunes:'
    )
    doc.add_picture('12_analisis_trayectorias.png', width=Inches(6.5))
    doc.add_paragraph()

doc.add_heading('5.4. Interpretación del Análisis de Trayectorias', 2)

if 'df_trayectorias' in locals():
    estables = df_trayectorias[df_trayectorias['categoria'].str.contains('ESTABLE')]
    estables_pct = (len(estables) / len(df_trayectorias)) * 100

    interpretacion_traj = [
        f'Estabilidad: {estables_pct:.1f}% de municipalidades mantienen el mismo cluster',
        f'Dinamismo: {100-estables_pct:.1f}% experimentan al menos un cambio de cluster',
        'Patrones dominantes: Las trayectorias estables son las más frecuentes',
        'Movilidad: Existe movilidad significativa entre clusters adyacentes',
        'Oportunidades: Las municipalidades que mejoran pueden servir de modelo de buenas prácticas'
    ]

    for interp in interpretacion_traj:
        doc.add_paragraph(interp, style='List Bullet')

doc.add_page_break()

# ============================================================
# COMPARACIÓN ENTRE METODOLOGÍAS
# ============================================================
print("🔬 Agregando comparación entre metodologías...")

doc.add_heading('6. Comparación entre las Tres Metodologías', 1)

doc.add_paragraph(
    'Cada metodología proporciona una perspectiva complementaria del desempeño presupuestal municipal. '
    'A continuación se presenta un análisis comparativo de sus características, ventajas y aplicaciones.'
)

doc.add_heading('6.1. Características Comparativas', 2)

# Tabla comparativa
comp_table = doc.add_table(rows=6, cols=4)
comp_table.style = 'Medium Grid 1 Accent 1'

headers = comp_table.rows[0].cells
headers[0].text = 'Aspecto'
headers[1].text = 'Estático'
headers[2].text = 'Panel'
headers[3].text = 'Trayectorias'

aspectos = [
    ['Enfoque', 'Global agregado', 'Temporal por año', 'Seguimiento individual'],
    ['Dimensión Temporal', 'Ignorada', 'Capturada año a año', 'Rastreada completa'],
    ['Nivel de Análisis', 'Poblacional', 'Anual', 'Individual'],
    ['Output Principal', '3 clusters globales', '3 clusters × 3 años', 'Categorías de trayectoria'],
    ['Aplicación', 'Benchmarking', 'Políticas anuales', 'Intervenciones focalizadas']
]

for i, row_data in enumerate(aspectos, 1):
    row = comp_table.rows[i].cells
    for j, cell_data in enumerate(row_data):
        row[j].text = cell_data

doc.add_heading('6.2. Similitudes entre Métodos', 2)

similitudes = [
    'Los tres métodos identifican consistentemente 3 clusters principales',
    'Los indicadores presupuestales muestran patrones de agrupamiento similares',
    'Las distribuciones son relativamente estables entre enfoques',
    'Todos confirman la existencia de grupos diferenciados de desempeño',
    'La validación estadística es consistente entre metodologías'
]

for sim in similitudes:
    doc.add_paragraph(sim, style='List Bullet')

doc.add_heading('6.3. Diferencias y Complementariedad', 2)

diferencias = [
    'Estático: Proporciona visión panorámica pero pierde información temporal',
    'Panel: Captura evolución pero no rastrea individuos específicos',
    'Trayectorias: Identifica patrones individuales pero requiere datos completos',
    'Los tres métodos se complementan para análisis integral',
    'Cada método responde a diferentes preguntas de política pública'
]

for dif in diferencias:
    doc.add_paragraph(dif, style='List Bullet')

doc.add_heading('6.4. Visualización Comparativa', 2)

if os.path.exists('13_comparacion_metodos.png'):
    doc.add_paragraph('Comparación visual de resultados entre las tres metodologías:')
    doc.add_picture('13_comparacion_metodos.png', width=Inches(6.5))
    doc.add_paragraph()

doc.add_heading('6.5. Recomendaciones de Uso por Metodología', 2)

recomendaciones_metodos = [
    'Estático: Ideal para reportes generales, clasificación inicial, y comunicación a audiencias no técnicas',
    'Panel: Recomendado para análisis presupuestales anuales, evaluación de políticas temporales, y seguimiento de indicadores',
    'Trayectorias: Esencial para identificar casos de éxito/fracaso, diseño de intervenciones focalizadas, y estudios longitudinales'
]

for rec in recomendaciones_metodos:
    doc.add_paragraph(rec, style='List Bullet')

doc.add_page_break()

# ============================================================
# CONCLUSIONES INTEGRADAS
# ============================================================
print("📝 Agregando conclusiones...")

doc.add_heading('7. Conclusiones y Recomendaciones', 1)

doc.add_heading('Hallazgos Principales', 2)

conclusiones = [
    'Los tres métodos confirman la existencia de 3 clusters principales de desempeño presupuestal',
    'Análisis Estático: Proporciona clasificación robusta con validación estadística rigurosa (Silhouette, Calinski-Harabasz)',
    'Análisis de Panel: Revela estabilidad estructural con evolución gradual de centroides 2022-2024',
    'Análisis de Trayectorias: Identifica alta proporción de municipalidades estables y patrones de movilidad significativos',
    'Existe complementariedad entre los tres enfoques para análisis integral del desempeño municipal',
    'Los indicadores muestran patrones consistentes de correlación que justifican el agrupamiento',
    'La variabilidad en desempeño presupuestal sugiere necesidad de políticas diferenciadas'
]

for conclusion in conclusiones:
    p = doc.add_paragraph(conclusion, style='List Bullet')

doc.add_heading('Recomendaciones Estratégicas', 2)

recomendaciones = [
    'Utilizar Análisis Estático para benchmarking y clasificación general de municipalidades',
    'Aplicar Análisis de Panel para monitoreo anual y ajuste de políticas presupuestales',
    'Emplear Análisis de Trayectorias para identificar municipalidades que requieren intervención',
    'Implementar políticas diferenciadas según cluster de pertenencia',
    'Estudiar municipalidades en trayectoria "MEJORANDO" como casos de buenas prácticas',
    'Fortalecer capacidades en municipalidades con trayectorias "EMPEORANDO" o de bajo desempeño',
    'Promover intercambio de experiencias entre municipalidades del mismo cluster',
    'Realizar seguimiento continuo con los tres enfoques para capturar diferentes dimensiones',
    'Profundizar análisis en clusters con alta variabilidad interna'
]

for rec in recomendaciones:
    p = doc.add_paragraph(rec, style='List Bullet')

doc.add_page_break()

# ============================================================
# ANEXOS
# ============================================================
print("📎 Agregando anexos...")

doc.add_heading('8. Anexos', 1)

doc.add_heading('Anexo A: Metodología Detallada', 2)

doc.add_paragraph(
    'K-Means Clustering: Algoritmo de particionamiento que agrupa observaciones en k clusters, '
    'donde cada observación pertenece al cluster con la media más cercana. Se utiliza el método del codo '
    'para determinar el número óptimo de clusters.'
)

doc.add_paragraph(
    'Análisis de Panel: Aplicación repetida de clustering K-Means para cada periodo temporal, '
    'permitiendo observar evolución de centroides y distribuciones año a año.'
)

doc.add_paragraph(
    'Análisis de Trayectorias: Seguimiento longitudinal de asignaciones de cluster para cada unidad, '
    'identificando patrones de estabilidad, mejora, deterioro o fluctuación.'
)

doc.add_paragraph(
    'Análisis de Componentes Principales (PCA): Técnica de reducción dimensional que transforma '
    'variables correlacionadas en componentes principales no correlacionados, facilitando la visualización '
    'de estructuras de alta dimensionalidad.'
)

doc.add_paragraph(
    'StandardScaler: Normalización de datos que estandariza las características eliminando la media '
    'y escalando a varianza unitaria. Esencial para K-Means ya que el algoritmo es sensible a la escala.'
)

doc.add_heading('Anexo B: Métricas de Validación', 2)
doc.add_paragraph('Las siguientes métricas fueron utilizadas para validar la calidad del clustering:')

metricas_desc = [
    'Silhouette Score: Mide qué tan similar es un objeto a su propio cluster comparado con otros clusters. Rango: [-1, 1]. Valores cercanos a 1 indican clustering apropiado.',
    'Calinski-Harabasz Index: Ratio de suma de dispersión entre clusters y dentro de clusters. Mayor valor indica clusters mejor definidos.',
    'Davies-Bouldin Index: Promedio de similitud entre clusters. Menor valor indica mejor separación entre clusters.',
    'Inercia (WCSS): Suma de distancias cuadradas de muestras al centroide más cercano. Menor valor indica clusters más compactos.'
]

for desc in metricas_desc:
    doc.add_paragraph(desc, style='List Bullet')

doc.add_heading('Anexo C: Archivos Generados', 2)
doc.add_paragraph('Los siguientes archivos fueron generados durante el análisis:')

archivos = [
    'DATASETS CSV:',
    '  - dataset_completo_con_clusters.csv - Dataset original con asignaciones',
    '  - resumen_cluster_estatico.csv - Estadísticas por cluster',
    '',
    'GRÁFICOS EXPLORATORIOS (300 DPI):',
    '  - 01_distribucion_indicadores.png - Histogramas',
    '  - 02_boxplots_indicadores.png - Análisis de variabilidad',
    '  - 03_scatter_relaciones.png - Relaciones entre variables',
    '  - 04_matriz_correlacion.png - Mapa de calor',
    '  - 05_evolucion_temporal.png - Tendencias 2022-2024',
    '',
    'ANÁLISIS ESTÁTICO (300 DPI):',
    '  - metodo_codo.png - Determinación K óptimo',
    '  - clusters_pca_visualization.png - Proyección PCA',
    '  - metricas_validacion_clustering.png - Validación estadística',
    '  - 06-10_*.png - Visualizaciones por cluster (5 gráficos)',
    '',
    'ANÁLISIS DE PANEL (300 DPI):',
    '  - 11_analisis_panel_evolucion.png - Evolución temporal de clusters',
    '',
    'ANÁLISIS DE TRAYECTORIAS (300 DPI):',
    '  - 12_analisis_trayectorias.png - Matrices de transición y categorías',
    '',
    'COMPARACIÓN DE MÉTODOS (300 DPI):',
    '  - 13_comparacion_metodos.png - Análisis comparativo de los 3 enfoques'
]

for archivo in archivos:
    if archivo.startswith(' '):
        doc.add_paragraph(archivo)
    elif archivo == '':
        doc.add_paragraph()
    else:
        p = doc.add_paragraph(archivo)
        p.bold = True

# ============================================================
# PIE DE PÁGINA
# ============================================================
footer = doc.sections[0].footer
footer_para = footer.paragraphs[0]
footer_para.text = f"Análisis generado automáticamente | {datetime.now().strftime('%d/%m/%Y %H:%M')} | Tres Metodologías Complementarias"
footer_para.alignment = WD_ALIGN_PARAGRAPH.CENTER

# ============================================================
# GUARDAR DOCUMENTO
# ============================================================
print("\n💾 Guardando documento Word...")

doc_filename = f'Informe_Clustering_Municipal_{datetime.now().strftime("%Y%m%d_%H%M%S")}.docx'
doc.save(doc_filename)

print(f"✅ Documento Word generado: {doc_filename}")
print(f"📊 Tamaño: {os.path.getsize(doc_filename) / 1024:.2f} KB")

# Copiar a carpeta results
if os.path.exists('results'):
    import shutil
    shutil.copy(doc_filename, f'results/{doc_filename}')
    print(f"✓ Copia guardada en: results/{doc_filename}")

print("\n" + "=" * 60)
print("✓ DOCUMENTO WORD COMPLETADO")
print("=" * 60)
print(f"\n📄 Archivo: {doc_filename}")
print("\n📋 Contenido del documento:")
print("   ✓ Portada con información de las 3 metodologías")
print("   ✓ Resumen ejecutivo integrado")
print("   ✓ Estadísticas descriptivas con tablas")
print("   ✓ Gráficos exploratorios individuales (5 visualizaciones)")
print("   ✓ MÉTODO 1: Análisis Estático K-Means completo")
print("   ✓ MÉTODO 2: Análisis de Panel temporal")
print("   ✓ MÉTODO 3: Análisis de Trayectorias")
print("   ✓ Comparación detallada entre los 3 métodos")
print("   ✓ Conclusiones y recomendaciones integradas")
print("   ✓ Anexos metodológicos completos")
print(f"   ✓ Total: 16 gráficos en 300 DPI")
print("\n🎉 ¡Documento completo con las 3 metodologías listo!")

# Descargar en Google Colab
try:
    from google.colab import files
    print("\n📥 Descargando documento en Google Colab...")
    files.download(doc_filename)
    print("✓ Descarga iniciada")
except:
    print(f"\nℹ️ Para descargar: busque el archivo '{doc_filename}' en el explorador de archivos")


In [ ]:
# @title 11. Pruebas y Validación de Gráficos 🧪
import os
import matplotlib.pyplot as plt

print("=" * 60)
print("PRUEBAS Y VALIDACIÓN DE GRÁFICOS GENERADOS")
print("=" * 60)

# Lista de archivos de gráficos esperados
graficos_esperados = [
    'visualizaciones_exploratorias.png',
    'metodo_codo.png',
    'clusters_pca_visualization.png',
    'metricas_validacion_clustering.png',
    'analisis_panel.png',
    'analisis_trayectorias.png'
]

print("\n📊 Verificando gráficos generados:")
graficos_encontrados = []
graficos_faltantes = []

for grafico in graficos_esperados:
    if os.path.exists(grafico):
        size_kb = os.path.getsize(grafico) / 1024
        print(f"✓ {grafico} - {size_kb:.2f} KB")
        graficos_encontrados.append(grafico)
    else:
        print(f"✗ {grafico} - NO ENCONTRADO")
        graficos_faltantes.append(grafico)

# Resumen de validación
print("\n" + "=" * 60)
print("RESUMEN DE VALIDACIÓN")
print("=" * 60)
print(f"✓ Gráficos encontrados: {len(graficos_encontrados)}/{len(graficos_esperados)}")
print(f"✗ Gráficos faltantes: {len(graficos_faltantes)}")

if graficos_faltantes:
    print("\n⚠️ ADVERTENCIA: Algunos gráficos no se generaron correctamente.")
    print("   Asegúrese de ejecutar todas las celdas anteriores.")
    print(f"   Faltantes: {', '.join(graficos_faltantes)}")
else:
    print("\n🎉 ¡Todos los gráficos se generaron correctamente!")

# Prueba de renderizado: crear un gráfico de prueba simple
print("\n🔬 Ejecutando prueba de renderizado...")
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot([1, 2, 3, 4], [1, 4, 2, 3], 'o-', linewidth=2, markersize=8)
ax.set_title('Gráfico de Prueba', fontsize=14, fontweight='bold')
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.grid(alpha=0.3)
plt.savefig('test_plot.png', dpi=150, bbox_inches='tight')
plt.close()

if os.path.exists('test_plot.png'):
    print("✓ Prueba de renderizado exitosa")
    os.remove('test_plot.png')  # Limpiar archivo de prueba
else:
    print("✗ Error en prueba de renderizado")

print("\n" + "=" * 60)
print("✓ Validación completada")
print("=" * 60)

In [ ]:
# @title 9. Resumen Final y Exportación de Resultados 📦
import pandas as pd
import os

print("=" * 60)
print("RESUMEN FINAL DEL ANÁLISIS")
print("=" * 60)

# Crear DataFrame con todos los resultados
df_resultados = df.copy()

# Agregar el nuevo clustering estático si existe
if 'df_clustered' in locals() and 'cluster_estatico_nuevo' in df_clustered.columns:
    # Fusionar con df_resultados usando el índice
    df_resultados = df_resultados.join(df_clustered['cluster_estatico_nuevo'], how='left')

print("\n📊 RESUMEN DE CLUSTERS:")
print(f"\n1. Total de municipalidades analizadas: {len(df)}")

# Verificar si tenemos el clustering nuevo
if 'cluster_estatico_nuevo' in df_resultados.columns:
    n_clusters = df_resultados['cluster_estatico_nuevo'].nunique()
    print(f"\n2. Clustering K-Means: {n_clusters} clusters identificados")
else:
    print("\n⚠️ Ejecute la celda de clustering (Celda 5) primero")

# Estadísticas por indicador
print(f"\n3. Estadísticas Generales de Indicadores:")
print("\n   Indicador de Ejecución (ind_eje):")
print(f"   - Media: {df['ind_eje'].mean():.4f}")
print(f"   - Mediana: {df['ind_eje'].median():.4f}")
print(f"   - Desv. Estándar: {df['ind_eje'].std():.4f}")

print("\n   Proporción Inversión Municipal (propim):")
print(f"   - Media: {df['propim'].mean():.4f}")
print(f"   - Mediana: {df['propim'].median():.4f}")
print(f"   - Desv. Estándar: {df['propim'].std():.4f}")

print("\n   Proporción Inversión (proinv):")
print(f"   - Media: {df['proinv'].mean():.4f}")
print(f"   - Mediana: {df['proinv'].median():.4f}")
print(f"   - Desv. Estándar: {df['proinv'].std():.4f}")

# Crear directorio para resultados
os.makedirs('results', exist_ok=True)

# Exportar DataFrames
print("\n💾 Exportando resultados...")

# 1. Dataset completo con clusters
df_resultados.to_csv('results/dataset_completo_con_clusters.csv', index=False)
print("✓ Dataset completo exportado")

# 2. Resumen por cluster estático (solo si existe el clustering nuevo)
if 'cluster_estatico_nuevo' in df_resultados.columns:
    # Filtrar datos con cluster asignado
    df_con_cluster = df_resultados.dropna(subset=['cluster_estatico_nuevo'])
    
    if len(df_con_cluster) > 0:
        resumen_estatico = df_con_cluster.groupby('cluster_estatico_nuevo')[['ind_eje', 'propim', 'proinv']].agg(['mean', 'median', 'std', 'count'])
        resumen_estatico.to_csv('results/resumen_cluster_estatico.csv')
        print("✓ Resumen cluster estático exportado")
    else:
        print("⚠️ No hay datos con cluster asignado")

print("\n📁 Archivos generados en directorio 'results/':")
for file in sorted(os.listdir('results')):
    file_path = os.path.join('results', file)
    size_kb = os.path.getsize(file_path) / 1024
    print(f"   - {file} ({size_kb:.2f} KB)")

print("\n" + "=" * 60)
print("✓ Exportación de resultados completada")
print("=" * 60)

In [ ]:
# @title 5. Clustering Estático con K-Means 🎯
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import numpy as np

print("=" * 60)
print("ANÁLISIS DE CLUSTERING ESTÁTICO (K-MEANS)")
print("=" * 60)

# Preparar datos para clustering
indicadores = ['ind_eje', 'propim', 'proinv']
X = df[indicadores].copy()

# Eliminar filas con valores faltantes
X_clean = X.dropna()
print(f"\n📊 Datos para clustering: {X_clean.shape[0]} municipalidades")

# Normalizar datos
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_clean)

# Método del codo para determinar número óptimo de clusters
print("\n🔍 Calculando método del codo...")
inertias = []
K_range = range(2, 11)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertias.append(kmeans.inertia_)

# Visualizar método del codo
plt.figure(figsize=(10, 6))
plt.plot(K_range, inertias, 'bo-', linewidth=2, markersize=8)
plt.xlabel('Número de Clusters (k)', fontsize=12)
plt.ylabel('Inercia (Within-Cluster Sum of Squares)', fontsize=12)
plt.title('Método del Codo para Determinar K Óptimo', fontsize=14, fontweight='bold')
plt.grid(alpha=0.3)
plt.savefig('metodo_codo.png', dpi=300, bbox_inches='tight')
print("✓ Gráfico del método del codo guardado")
plt.show()

# Aplicar K-Means con k óptimo (usaremos k=3 como en los datos originales)
k_optimo = 3
print(f"\n🎯 Aplicando K-Means con k={k_optimo}...")

kmeans = KMeans(n_clusters=k_optimo, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_scaled)

# Agregar clusters al DataFrame
df_clustered = X_clean.copy()
df_clustered['cluster_estatico_nuevo'] = clusters

# Análisis de clusters
print(f"\n📈 DISTRIBUCIÓN DE CLUSTERS:")
print(df_clustered['cluster_estatico_nuevo'].value_counts().sort_index())

print(f"\n📊 CENTROIDES DE CLUSTERS (valores normalizados):")
centroides = kmeans.cluster_centers_
centroides_df = pd.DataFrame(centroides, columns=indicadores)
centroides_df.index = [f'Cluster {i}' for i in range(k_optimo)]
print(centroides_df)

# Desnormalizar centroides para interpretación
print(f"\n📊 CENTROIDES DE CLUSTERS (valores originales):")
centroides_orig = scaler.inverse_transform(centroides)
centroides_orig_df = pd.DataFrame(centroides_orig, columns=indicadores)
centroides_orig_df.index = [f'Cluster {i}' for i in range(k_optimo)]
print(centroides_orig_df)

# Características promedio por cluster
print(f"\n📊 ESTADÍSTICAS POR CLUSTER:")
for i in range(k_optimo):
    print(f"\n--- CLUSTER {i} ---")
    cluster_data = df_clustered[df_clustered['cluster_estatico_nuevo'] == i][indicadores]
    print(cluster_data.describe())

print("\n" + "=" * 60)
print("✓ Análisis de clustering estático completado")
print("=" * 60)

In [ ]:
# @title 3. Exploración y Análisis Descriptivo de Datos 📊
import pandas as pd
import numpy as np

print("=" * 60)
print("ANÁLISIS EXPLORATORIO DE DATOS")
print("=" * 60)

# Verificar valores faltantes
print("\n1️⃣ VALORES FALTANTES:")
print(df.isnull().sum())

# Información del dataset
print("\n2️⃣ INFORMACIÓN DEL DATASET:")
print(f"   Total de registros: {len(df):,}")
print(f"   Años en el dataset: {sorted(df['year'].unique())}")
print(f"   Municipalidades únicas: {df['ejecutora_nombre'].nunique():,}")
print(f"   Registros por año:")
for year in sorted(df['year'].unique()):
    count = (df['year'] == year).sum()
    print(f"      {year}: {count:,} registros")

# Estadísticas descriptivas de los indicadores principales
print("\n3️⃣ ESTADÍSTICAS DESCRIPTIVAS - INDICADORES PRINCIPALES:")
indicadores_brutos = ['ind_eje', 'propim', 'proinv']
print(df[indicadores_brutos].describe())

# Agregar datos por año para análisis
print("\n4️⃣ PROMEDIOS POR AÑO:")
print(df.groupby('year')[indicadores_brutos].mean())

# Correlación entre indicadores
print("\n5️⃣ MATRIZ DE CORRELACIÓN:")
correlacion = df[indicadores_brutos].corr()
print(correlacion)

print("\n" + "=" * 60)
print("✓ Análisis exploratorio completado")
print("=" * 60)